## Cell 01 — Scientific scope, experiments E1–E10, and non-negotiable evidence rules

**Purpose.** Reanalyse the 256 original observations (54 cryogenic, 160 room temperature, 42 elevated temperature) for the journal revision. This is a new computational study; it does not validate authors' laboratory measurements unless independent records are supplied. Source labels are **author-confirmed, identical to earlier random suggestion files, and NOT independently paper-to-row matched**; the manuscript's 259-vs-256 count difference remains open.

**Experiment ledger.** E1 five regressors (ET, GB, XGBoost, RF, SVR); E2 no augmentation versus training-only 1%/historical-magnitude 3% Gaussian *input perturbation sensitivity* with two train-time copies (3× training rows); E3 publication/material grouping; E4 evidence-screened epoxy versus mixed literature; E5 physical effective-medium baselines; E6 temperature-regime comparisons; E7 honest held-out conformal intervals; E8 independent external validation/OOD; E9 physical plausibility; E10 Fourier-law/uncertainty/calibration **only where measurements exist**. Every unsupported task emits a documented `NOT_ESTIMABLE`/`BLOCKED` record.

The notebook writes individual CSV/JSON/PNG files throughout execution, stores final fitted models and a consolidated ZIP. It **never** uses metadata or target as a predictor, imputes a measured room temperature, silently changes loading units, assigns new source IDs, or substitutes a random split for an infeasible grouped split.

**Reviewer mapping.** The package contains the 40-item reviewer issue register and a code-generated status table; software outputs do not replace experimental evidence or manuscript rewriting.
**Scientific warning.** The source labels are currently identical to previously randomly assigned placeholders. Group-isolation by these labels is a useful software stress test, **not** a validated leave-publication-out scientific generalization estimate. This notebook cannot establish source identity, loading units, experimental provenance, or independent validation from the numerical fields alone. Before a manuscript claim, independently verify the underlying records.


In [1]:
print('BLOCK 2 — Epoxy thermal-conductivity reviewer-response computations')
print('Modes: grouped provisional evaluation, physics/measurement gates, honest not-estimable results.')
print('The input source labels were author confirmed but not independently matched to paper tables.')

BLOCK 2 — Epoxy thermal-conductivity reviewer-response computations
Modes: grouped provisional evaluation, physics/measurement gates, honest not-estimable results.
The input source labels were author confirmed but not independently matched to paper tables.


## Cell 03 — Runtime settings, CPU budget, and explicit smoke-test option

Use CPU. The default `TC_RUN_MODE=full` executes five regressors, three augmentation settings (none, 1% and historical-magnitude 3%), leave-one-publication-out nested CV where possible, downstream reports and model serialization. To sanity-test quickly before Kaggle's final version, set `TC_RUN_MODE=smoke` in Cell 04 (or as environment variable): two models, two outer folds, and one tuning candidate; smoke outputs are never final publication results. Change `INPUT_DIR` only if your Kaggle dataset has a different path. The final working files must have the exact Kaggle-ready filenames.

Outer folds never train on the held-out publication. Inner folds tune *only within outer training groups*. The 42 elevated-temperature rows have one publication and are *not* assigned invented folds.

In [2]:
import os, sys, json, csv, math, hashlib, shutil, platform, warnings, itertools, time
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
from scipy import stats
from scipy.optimize import brentq
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.neural_network import MLPRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import sklearn
SEED = 20261107
RUN_MODE = os.environ.get('TC_RUN_MODE', 'full').strip().lower()   # full or smoke
assert RUN_MODE in ('full','smoke')
INPUT_DIR = Path("/kaggle/input/datasets/shanmukhadattaboda/thermal-conductivity-datasets")
CPU_JOBS = max(1, min(int(os.environ.get('TC_CPU_JOBS','2')), 4))
SEARCH_ITER = int(os.environ.get('TC_SEARCH_ITER', '4' if RUN_MODE=='full' else '1'))
assert 1 <= SEARCH_ITER <= 30
INNER_SPLITS = 3 if RUN_MODE=='full' else 2
MAX_OUTER = None if RUN_MODE=='full' else 2
AUGMENTATION_LEVEL = 0.01
HISTORICAL_NOISE_LEVEL = 0.03
INCLUDE_HISTORICAL_3PCT = os.environ.get('TC_INCLUDE_HISTORICAL_3PCT','1') == '1'
RANDOM_SPLITS_PERMITTED = False
AUGMENTATION_COPIES = 2
CONFORMAL_ALPHA = 0.10
RUN_SHAP = RUN_MODE=='full'
SAVE_OUTER_MODELS = True
np.random.seed(SEED)
# Avoid suppressing all FutureWarnings; log library-version sensitivity in the run manifest.
print({'mode':RUN_MODE,'input_dir':str(INPUT_DIR),'search_iter':SEARCH_ITER,'cpu_jobs':CPU_JOBS,'seed':SEED,'historical_3pct':INCLUDE_HISTORICAL_3PCT})
EXTRA_BASELINES = os.environ.get('TC_EXTRA_BASELINES', '1' if RUN_MODE=='full' else '0') == '1'


{'mode': 'full', 'input_dir': '/kaggle/input/datasets/shanmukhadattaboda/thermal-conductivity-datasets', 'search_iter': 4, 'cpu_jobs': 2, 'seed': 20261107, 'historical_3pct': True}


## Cell 05 — Reproducibility, output folder tree, and safe artifact writers

Create a **fresh, clean** `/kaggle/working/TC_BLOCK2_RESULTS` on each run to avoid old files being mistaken for current results. Every output is registered with stage, relative path, and SHA-256. Raw data will not be edited. If run outside Kaggle, use the current working directory. The final ZIP is `/kaggle/working/TC_BLOCK2_RESULTS.zip` on Kaggle.

In [3]:
WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = WORKING/'TC_BLOCK2_RESULTS'
assert OUT.name=='TC_BLOCK2_RESULTS'
if OUT.exists():
    assert OUT.is_dir() and OUT.parent==WORKING and not OUT.is_symlink()
    shutil.rmtree(OUT)
for d in ['00_config','01_audit','02_preprocessing','03_splits','04_models','05_nested_cv','06_stats','07_interpretation','08_external','09_physics','10_uncertainty','11_reports','12_figures','13_packages']:
    (OUT/d).mkdir(parents=True,exist_ok=True)
ARTIFACTS=[]
def artifact(path,stage):
    p=Path(path)
    if not p.exists(): raise FileNotFoundError(p)
    ARTIFACTS.append({'stage':stage,'relative_path':p.relative_to(OUT).as_posix(), 'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
    return p
def save_csv(frame, stage, name, index=False):
    p=OUT/stage/name
    pd.DataFrame(frame).to_csv(p,index=index)
    artifact(p,stage); return p
def save_json(value,stage,name):
    p=OUT/stage/name; p.write_text(json.dumps(value,indent=2,default=str,allow_nan=False),encoding='utf-8');artifact(p,stage);return p
def save_figure(stage,name):
    p=OUT/stage/name;plt.tight_layout();plt.savefig(p,dpi=165,bbox_inches='tight');plt.close();artifact(p,stage);return p
STATES=[]
def status(stage, experiment, state, reason, n=0):
    STATES.append({'stage':stage,'experiment':experiment,'status':state,'details':reason,'n_records':int(n)})
    save_csv(STATES,'11_reports','experiment_status_running.csv')
run_info={'generated_utc':datetime.now(timezone.utc).isoformat(),'python':platform.python_version(),'sklearn':sklearn.__version__,'numpy':np.__version__,'pandas':pd.__version__,'seed':SEED,'run_mode':RUN_MODE,'cpu_jobs':CPU_JOBS,'search_iter':SEARCH_ITER,'input_root_requested':str(INPUT_DIR),'augmentation_input_only_sd_relative':AUGMENTATION_LEVEL,'augmentation_train_copies':AUGMENTATION_COPIES,'historical_experiment_not_exact_reproduction':True,'historical_implemented_noise_sd_relative':HISTORICAL_NOISE_LEVEL,'historical_3pct_experiment_enabled':INCLUDE_HISTORICAL_3PCT,'run_mode_full_required_for_final':True,'xgboost_version':'PENDING_IMPORT','scipy_version':getattr(__import__('scipy'),'__version__','unknown'),'matplotlib_version':matplotlib.__version__,'joblib_version':joblib.__version__}
save_json(run_info,'00_config','runtime_config.json')
print('Output root:',OUT)

Output root: /kaggle/working/TC_BLOCK2_RESULTS


## Cell 07 — Input discovery: precisely three finalized Block 1 datasets

The required filenames are `CRYOGENIC_KAGGLE_READY.csv`, `ROOM_KAGGLE_READY.csv`, and `HIGH_KAGGLE_READY.csv`. First check the configured Kaggle directory, then `/kaggle/input`. If duplicated names occur in multiple datasets, fail rather than choose an arbitrary file; set `TC_INPUT_DIR` to the intended dataset. Outside Kaggle, the accompanying `input_data/` works automatically. Preserve the exact file hash and row order.

In [4]:
FILENAMES={'cryogenic':'CRYOGENIC_KAGGLE_READY.csv','room':'ROOM_KAGGLE_READY.csv','high':'HIGH_KAGGLE_READY.csv'}
def locate(name):
    candidates=[INPUT_DIR/name,Path.cwd()/'input_data'/name,Path.cwd()/name,Path('/mnt/data/TC_BLOCK1_AUTHOR_CONFIRMED')/name]
    for p in candidates:
        if p.is_file():return p
    roots=[Path('/kaggle/input')]
    found=[]
    for root in roots:
        if root.exists():found.extend(root.rglob(name))
    found=list(dict.fromkeys(found))
    if len(found)==1:return found[0]
    if not found: raise FileNotFoundError(f'Missing {name}; attach all three FINAL Block 1 CSVs, not source-mapping drafts.')
    raise RuntimeError(f'Multiple files named {name}; specify TC_INPUT_DIR: {found}')
FILES={reg:locate(name) for reg,name in FILENAMES.items()}
RAW={reg:pd.read_csv(f,keep_default_na=True) for reg,f in FILES.items()}
sha=[{'regime':reg,'filename':p.name,'path':str(p),'rows':len(RAW[reg]),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for reg,p in FILES.items()]
save_csv(sha,'01_audit','input_file_hashes.csv')
print(pd.DataFrame(sha)[['regime','filename','rows','sha256']].to_string(index=False))

   regime                   filename  rows                                                           sha256
cryogenic CRYOGENIC_KAGGLE_READY.csv    54 388bec35201c269a097110bb6a7770fd283b97dd176415c50c23d3c6e972e34f
     room      ROOM_KAGGLE_READY.csv   160 7d0e2aff60893058f8e5ed48d853239caffc6110c7d399e5fcbd06c333a27741
     high      HIGH_KAGGLE_READY.csv    42 2507c792280202e7ec426ea018daac5d1e556b0ee15bf135fa1fdda5d9722a09


## Cell 09 — Strict schema, target validity, and no silent row modifications

Require the exact original counts and unique `orig_id`s. For every original observation, `base_tc_w_mk`, `filler_tc_w_mk`, `filler_loading_as_recorded` and `composite_tc_w_mk` must be numeric and finite. Temperatures are recorded for cryogenic/high, but **not** invented for room. `publication_id` must exist, because source-grouped CV depends on it. Validation failures stop the notebook and write a diagnostic CSV.

In [5]:
CORE=['orig_id','temperature_regime','base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded','composite_tc_w_mk','publication_id']
EXPECTED={'cryogenic':54,'room':160,'high':42}
validation=[]
for regime, df in RAW.items():
    miss=[c for c in CORE if c not in df.columns]
    validation.append({'regime':regime,'check':'required_columns','pass':not bool(miss),'details':','.join(miss)})
    validation.append({'regime':regime,'check':'original_count','pass':len(df)==EXPECTED[regime],'details':f'{len(df)} expected {EXPECTED[regime]}'})
    if miss: continue
    for c in ['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded','composite_tc_w_mk']:
        df[c]=pd.to_numeric(df[c],errors='coerce')
    valid_unique=df.orig_id.notna().all() and df.orig_id.astype(str).is_unique and df.orig_id.astype(str).str.match({'cryogenic':r'^CRY_\d+$','room':r'^ROO_\d+$','high':r'^HIG_\d+$'}[regime]).all()
    validation.append({'regime':regime,'check':'unique_observation_ids','pass':bool(valid_unique),'details':''})
    for col in ['base_tc_w_mk','filler_tc_w_mk','composite_tc_w_mk']:
        ok=np.isfinite(df[col]).all() and (df[col]>0).all()
        validation.append({'regime':regime,'check':'positive_finite_'+col,'pass':bool(ok),'details':f'invalid={int((~np.isfinite(df[col])|(df[col]<=0)).sum())}'})
    ok=np.isfinite(df.filler_loading_as_recorded).all() and (df.filler_loading_as_recorded>=0).all()
    validation.append({'regime':regime,'check':'finite_nonnegative_loading','pass':bool(ok),'details':''})
    if regime!='room':
        ok=np.isfinite(df.temperature_k_recorded).all() and (df.temperature_k_recorded>0).all()
        validation.append({'regime':regime,'check':'recorded_temperature','pass':bool(ok),'details':''})
    else:
        validation.append({'regime':regime,'check':'room_measured_temperature_absent','pass':bool(df.temperature_k_recorded.isna().all()),'details':'No invented per-row temperatures'})
    valid_pub=df.publication_id.notna().all() and (df.publication_id.astype(str).str.strip()!='').all()
    validation.append({'regime':regime,'check':'publication_id_present','pass':bool(valid_pub),'details':''})
save_csv(validation,'01_audit','schema_validation.csv')
fail=[x for x in validation if not x['pass']]
if fail: raise ValueError('Dataset validation failed. See 01_audit/schema_validation.csv: '+str(fail[:4]))
DATA=pd.concat([RAW[x].copy() for x in EXPECTED],ignore_index=True)
assert DATA.orig_id.astype(str).is_unique and len(DATA)==256
save_csv(DATA,'01_audit','all_256_original_rows_unchanged.csv')
print('PASS: exact 256 records; schema/identity/target/temperature rules verified.')
# Source labels are unverified; reject invalid or cross-regime labels before grouped CV.
for reg,d in RAW.items():
    allowed={'cryogenic':{f'REF_{x}' for x in range(19,24)},
             'room':{f'REF_{x}' for x in range(24,29)}, 'high':{'REF_29'}}[reg]
    observed=set(d.publication_id.astype(str).str.strip())
    if not observed.issubset(allowed):
        raise ValueError(f'{reg}: invalid bibliography labels {observed-allowed}; no fabricated groups permitted')
    if 'source_assignment_status' not in d:
        raise ValueError(f'{reg}: source_assignment_status required to prevent unqualified interpretation')
    if d.source_assignment_status.fillna('').astype(str).str.strip().eq('').any():
        raise ValueError(f'{reg}: source assignment status missing')
    for other in ['material_system_id','source_row_evidence','filler_loading_unit','source_measurement_type']:
        if other not in d: d[other]=np.nan
save_csv([{'regime':reg,'declared_publications':';'.join(sorted(set(df.publication_id.astype(str)))),
           'all_source_labels_independently_verified':False,
           'safe_claim':'EXPLORATORY_ONLY'} for reg,df in RAW.items()],
         '01_audit','source_label_trust_gate.csv')


PASS: exact 256 records; schema/identity/target/temperature rules verified.


PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/01_audit/source_label_trust_gate.csv')

## Cell 11 — Numeric quality, duplicates, conflicts, and two-constituent sanity checks

Record exact duplicates, same feature inputs with different measured targets, nonphysical or suspicious ranges and missing metadata. Constituent-bound departures are **flags** only: anisotropy, interfacial effects or source/unit errors could make a simple two-constituent assumption inapplicable. Do not automatically remove these rows. Export **row-level flags** and a summary.

In [6]:
FLAGS=[]
full_cols=['temperature_regime','base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded','composite_tc_w_mk']
for reg, d in RAW.items():
    feat=['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded']+(['temperature_k_recorded'] if reg!='room' else [])
    dd=d.copy()
    if dd[full_cols].duplicated(keep=False).any():
        for id_ in dd.loc[dd[full_cols].duplicated(keep=False),'orig_id']:FLAGS.append({'orig_id':id_,'regime':reg,'flag':'EXACT_NUMERIC_DUPLICATE','detail':'All recorded model fields identical'})
    for _,g in dd.groupby(feat,dropna=False):
        if len(g)>1 and g.composite_tc_w_mk.nunique()>1:
            for id_ in g.orig_id:FLAGS.append({'orig_id':id_,'regime':reg,'flag':'SAME_INPUT_DIFFERENT_TARGET','detail':','.join(map(str,g.composite_tc_w_mk.tolist()))})
    mask=(dd.composite_tc_w_mk<dd[['base_tc_w_mk','filler_tc_w_mk']].min(axis=1)) | (dd.composite_tc_w_mk>dd[['base_tc_w_mk','filler_tc_w_mk']].max(axis=1))
    for id_ in dd.loc[mask,'orig_id']:FLAGS.append({'orig_id':id_,'regime':reg,'flag':'OUTSIDE_CONSTITUENT_RANGE','detail':'Not an automatic exclusion'})
    for _,rr in dd.iterrows():
        if pd.isna(rr.get('filler_loading_unit')) or str(rr.get('filler_loading_unit','')).strip()=='':
            FLAGS.append({'orig_id':rr.orig_id,'regime':reg,'flag':'LOADING_UNIT_UNVERIFIED','detail':'Vol field retained as recorded; not converted'})
        if pd.isna(rr.get('source_row_evidence')) or str(rr.get('source_row_evidence','')).strip()=='':
            FLAGS.append({'orig_id':rr.orig_id,'regime':reg,'flag':'NO_PRIMARY_ROW_EVIDENCE','detail':'Source assigned by author, not source-figure/table independently confirmed'})
        if pd.isna(rr.get('source_measurement_type')) or str(rr.get('source_measurement_type','')).strip()=='':
            FLAGS.append({'orig_id':rr.orig_id,'regime':reg,'flag':'MEASUREMENT_TYPE_UNKNOWN','detail':'Cannot assert all targets experimental'})
flags=pd.DataFrame(FLAGS,columns=['orig_id','regime','flag','detail'])
save_csv(flags,'01_audit','row_quality_flags.csv')
save_csv(flags.groupby(['regime','flag']).size().reset_index(name='count'),'01_audit','quality_flags_summary.csv')
print(flags.groupby('flag').orig_id.nunique().to_string())

flag
LOADING_UNIT_UNVERIFIED        256
MEASUREMENT_TYPE_UNKNOWN       256
NO_PRIMARY_ROW_EVIDENCE        256
OUTSIDE_CONSTITUENT_RANGE        4
SAME_INPUT_DIFFERENT_TARGET      6


## Cell 13 — Publication provenance distributions and temperature-domain coverage

Count source groups in each regime and retain DOI/title/author-confirmed labels. Record source concentration and exact observed temperature ranges. The manuscript's **259** original points are not recovered by declaring 3 imaginary observations. Plot study distribution and actual temperature coverage; large unmeasured temperature gaps should not be reported as validated continuous thermal behavior.

In [7]:
prov=DATA.groupby(['temperature_regime','publication_id','source_doi','source_assignment_status'],dropna=False).size().reset_index(name='n_observations')
save_csv(prov,'01_audit','publication_distribution.csv')
coverage=[]
for reg,d in RAW.items():
    temp=d.temperature_k_recorded.dropna()
    coverage.append({'regime':reg,'n_rows':len(d),'n_publications':d.publication_id.nunique(),'n_actual_temperature_rows':len(temp),'min_recorded_k':float(temp.min()) if len(temp) else None,'max_recorded_k':float(temp.max()) if len(temp) else None,'temperature_nominal_only':reg=='room','source_evidence_rows':int(d.source_row_evidence.notna().sum()) if 'source_row_evidence' in d else 0})
save_csv(coverage,'01_audit','regime_coverage.csv')
fig,ax=plt.subplots(figsize=(10,4))
for reg,part in prov.groupby('temperature_regime'):
    ax.bar([f'{reg} {s}' for s in part.publication_id],part.n_observations,label=reg)
ax.set_ylabel('Original observations');ax.set_title('Declared publication coverage (author confirmed, not independently matched)');ax.tick_params(axis='x',rotation=55);save_figure('12_figures','publication_counts.png')
fig,ax=plt.subplots(figsize=(8,4))
for reg,d in RAW.items():
    t=d.temperature_k_recorded.dropna()
    if not t.empty:ax.scatter(t,[reg]*len(t),alpha=.65,s=23,label=reg)
ax.set_xlabel('Measured/recorded temperature (K)');ax.set_title('Recorded temperature coverage; room nominal 300 K is not plotted');save_figure('12_figures','temperature_coverage.png')
print(pd.DataFrame(coverage).to_string(index=False))

   regime  n_rows  n_publications  n_actual_temperature_rows  min_recorded_k  max_recorded_k  temperature_nominal_only  source_evidence_rows
cryogenic      54               5                         54             2.0           120.0                     False                     0
     room     160               5                          0             NaN             NaN                      True                     0
     high      42               1                         42           308.0           423.0                     False                     0


## Cell 15 — Loading units: evidence-gated conversions and invalid-physics register

The original column named `Vol` is not proof of volume fraction. Derive `filler_volume_fraction` **only** if the per-row loading unit is documented and valid. Weight-to-volume conversion additionally requires independently documented matrix/filler densities; none is assumed. This gate prevents Maxwell/Bruggeman/Hashin–Shtrikman calculations on unknown units. Machine learning can still use the numeric loading *as recorded*, explicitly as a provisional heterogeneous feature.

In [8]:
def as_volume_fraction(rec):
    raw=rec.get('filler_loading_as_recorded',np.nan)
    if not np.isfinite(raw):return np.nan,'INVALID_LOADING'
    unit=str(rec.get('filler_loading_unit','')).strip().lower().replace(' ','')
    # Never promote a user-typed unit to verified physics without documentary unit evidence.
    if not str(rec.get('loading_unit_evidence','')).strip() or str(rec.get('loading_unit_evidence','')).lower()=='nan':
        return np.nan,'UNIT_LABEL_LACKS_DOCUMENTARY_EVIDENCE' if unit and unit!='nan' else 'UNKNOWN_LOADING_UNIT'
    if unit in ['vol%','vol_percent','volume_percent','volume%','volpct']:
        return (raw/100,'VERIFIED_VOLUME_PERCENT') if 0<=raw<=100 else (np.nan,'INVALID_VOLUME_PERCENT')
    if unit in ['vol_fraction','volume_fraction','volume_fraction_0_1']:
        return (raw,'VERIFIED_VOLUME_FRACTION') if 0<=raw<=1 else (np.nan,'INVALID_VOLUME_FRACTION')
    if unit in ['wt%','weight_percent','weight%','wt_percent','wt_fraction']:
        wm=raw/100 if unit!='wt_fraction' else raw
        rho_m=pd.to_numeric(rec.get('matrix_density_kg_m3',np.nan),errors='coerce')
        rho_f=pd.to_numeric(rec.get('filler_density_kg_m3',np.nan),errors='coerce')
        densities_documented=bool(str(rec.get('density_evidence','')).strip()) and str(rec.get('density_evidence','')).lower()!='nan'
        if densities_documented and 0<=wm<=1 and np.isfinite(rho_m) and np.isfinite(rho_f) and rho_m>0 and rho_f>0:
            vf=(wm/rho_f)/((wm/rho_f)+((1-wm)/rho_m))
            return vf,'WEIGHT_TO_VOLUME_WITH_VERIFIED_DENSITIES'
        return np.nan,'WEIGHT_PERCENT_MISSING_VERIFIED_DENSITIES'
    return np.nan,'UNKNOWN_LOADING_UNIT'
vals=[as_volume_fraction(r) for r in DATA.to_dict('records')]
DATA['filler_volume_fraction']=[v[0] for v in vals]
DATA['volume_fraction_status']=[v[1] for v in vals]
save_csv(DATA[['orig_id','temperature_regime','filler_loading_as_recorded','filler_loading_unit','filler_volume_fraction','volume_fraction_status']],'02_preprocessing','loading_unit_gate.csv')
save_csv(DATA.groupby('volume_fraction_status').size().reset_index(name='n_rows'),'02_preprocessing','loading_unit_summary.csv')
assert DATA.filler_volume_fraction.dropna().between(0,1).all()
print(DATA.volume_fraction_status.value_counts().to_string())

volume_fraction_status
UNKNOWN_LOADING_UNIT    256


## Cell 17 — Dataset eligibility and mixed-material scientific screens

Define three populations separately: **all original observations** (heterogeneous exploratory benchmark), **study-level epoxy candidates** (advisory only), and **strict evidence-backed epoxy experimental** rows (requires explicit row evidence, verified volume unit, experimental measurement type and epoxy material context). Do not treat publication study material labels as proof of each row's chemical identity; do not drop non-epoxy observations by a guessed conductivity threshold.

In [9]:
DATA['study_level_epoxy_candidate']=DATA.epoxy_particle_training_screen_from_publication.astype(str).eq('epoxy_particle_candidate')
evidence=DATA.source_row_evidence.fillna('').astype(str).str.strip().ne('')
measured=DATA.source_measurement_type.fillna('').astype(str).str.lower().str.strip().isin(['experimental','measured','physical_measurement'])
verified_volume=DATA.filler_volume_fraction.notna()
material=DATA.get('matrix_material_verified',pd.Series('',index=DATA.index)).fillna('').astype(str).str.lower().str.strip().eq('epoxy')
material_proof=DATA.get('matrix_material_evidence',pd.Series('',index=DATA.index)).fillna('').astype(str).str.strip().ne('')
primary_verified=DATA.get('source_row_verification_status',pd.Series('',index=DATA.index)).fillna('').astype(str).str.strip().eq('PRIMARY_SOURCE_VERIFIED')
DATA['strict_epoxy_eligible']=DATA.study_level_epoxy_candidate & evidence & measured & verified_volume & material & material_proof & primary_verified
counts=DATA.groupby('temperature_regime').agg(original=('orig_id','size'),study_level_epoxy_candidate=('study_level_epoxy_candidate','sum'),strict_epoxy_eligible=('strict_epoxy_eligible','sum'),publication_groups=('publication_id','nunique')).reset_index()
save_csv(counts,'02_preprocessing','population_eligibility_counts.csv')
save_csv(DATA[['orig_id','temperature_regime','publication_id','study_level_epoxy_candidate','strict_epoxy_eligible','source_row_evidence','source_measurement_type','volume_fraction_status']],'02_preprocessing','eligibility_ledger.csv')
status('02_preprocessing','E4_STRICT_EPOXY','READY' if DATA.strict_epoxy_eligible.any() else 'NOT_ESTIMABLE','Strict experimental epoxy requires independent per-row evidence, verified units and experimental type',int(DATA.strict_epoxy_eligible.sum()))
print(counts.to_string(index=False))

temperature_regime  original  study_level_epoxy_candidate  strict_epoxy_eligible  publication_groups
         cryogenic        54                           10                      0                   5
              high        42                            0                      0                   1
              room       160                            0                      0                   5


## Cell 19 — Feature definitions and leakage-free train-time transformations

Model predictors are **only** `base_tc_w_mk`, `filler_tc_w_mk`, `filler_loading_as_recorded`, and—where actually recorded—`temperature_k_recorded`. `composite_tc_w_mk` is the target; `orig_id`, DOI, `publication_id`, material/paper labels, status and row evidence are metadata, never features. Each fitted pipeline scales the SVR data **inside its training fold**; no global preprocessing is fit on held-out observations.

In [10]:
FEATURES={'cryogenic':['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded'],'room':['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded'],'high':['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','temperature_k_recorded']}
TARGET='composite_tc_w_mk'
FEATURE_SCHEMAS=[]
for reg,d in RAW.items():
    for col in FEATURES[reg]:FEATURE_SCHEMAS.append({'regime':reg,'feature':col,'used_as_input':True,'nonmissing':int(d[col].notna().sum()),'training_only_transform':'SVR: StandardScaler within pipeline; trees: no scale'})
    FEATURE_SCHEMAS.append({'regime':reg,'feature':TARGET,'used_as_input':False,'nonmissing':len(d),'training_only_transform':'target only'})
    X=d[FEATURES[reg]].to_numpy(dtype=float)
    assert np.isfinite(X).all() and X.shape[1] == (3 if reg=='room' else 4)
save_csv(FEATURE_SCHEMAS,'02_preprocessing','feature_definitions.csv')
save_json({x:FEATURES[x] for x in FEATURES},'02_preprocessing','feature_set_manifest.json')
print(json.dumps(FEATURES,indent=2))

{
  "cryogenic": [
    "base_tc_w_mk",
    "filler_tc_w_mk",
    "filler_loading_as_recorded",
    "temperature_k_recorded"
  ],
  "room": [
    "base_tc_w_mk",
    "filler_tc_w_mk",
    "filler_loading_as_recorded"
  ],
  "high": [
    "base_tc_w_mk",
    "filler_tc_w_mk",
    "filler_loading_as_recorded",
    "temperature_k_recorded"
  ]
}


## Cell 21 — Grouped evaluation feasibility: publication versus material

Publication-level leave-one-source-out nested CV requires **at least three** independent publication groups: one held-out source and at least two for inner tuning. A material-grouped analysis similarly requires at least three nonblank `material_system_id` values. Elevated-temperature data have only **one** source; never silently replace its group CV with an IID random split. Output explicit feasibility decisions for all regimes and grouping modes.

In [11]:
FEASIBILITY=[]
for reg,d in RAW.items():
    for grouping,column in [('publication','publication_id'),('material','material_system_id')]:
        if column not in d: g=pd.Series(dtype=str)
        else:g=d[column].fillna('').astype(str).str.strip();g=g[g.ne('')]
        count=int(g.nunique())
        evidence_for_groups=(column!='material_system_id' or ('material_system_evidence' in d and d.material_system_evidence.fillna('').astype(str).str.strip().ne('').all()))
        eligible=count>=3 and len(g)==len(d) and evidence_for_groups
        FEASIBILITY.append({'regime':reg,'grouping':grouping,'n_distinct_nonblank_groups':count,'n_grouped_rows':len(g),'eligible_for_nested_group_cv':eligible,'reason':'At least 3 complete independent groups' if eligible else 'Insufficient groups or missing group ID; no IID fallback'})
save_csv(FEASIBILITY,'03_splits','group_cv_feasibility.csv')
for item in FEASIBILITY:
    if not item['eligible_for_nested_group_cv']:status('03_splits',f'E3_{item["grouping"]}_{item["regime"]}','NOT_ESTIMABLE',item['reason'],item['n_grouped_rows'])
print(pd.DataFrame(FEASIBILITY).to_string(index=False))

   regime    grouping  n_distinct_nonblank_groups  n_grouped_rows  eligible_for_nested_group_cv                                                   reason
cryogenic publication                           5              54                          True                   At least 3 complete independent groups
cryogenic    material                           0               0                         False Insufficient groups or missing group ID; no IID fallback
     room publication                           5             160                          True                   At least 3 complete independent groups
     room    material                           0               0                         False Insufficient groups or missing group ID; no IID fallback
     high publication                           1              42                         False Insufficient groups or missing group ID; no IID fallback
     high    material                           0               0                 

## Cell 23 — Outer leave-one-publication-out folds and original-row split manifests

For each viable regime, create **one held-out source per outer fold**, using only original observations. Export train/test row IDs for every fold. Full mode evaluates all five outer sources; smoke mode truncates to two folds (labelled non-final). The tested groups must not intersect training groups. All later augmentation is applied by estimator `.fit()` **after** this split.

In [12]:
SPLITS={}; split_rows=[]
for reg,d in RAW.items():
    if not next(z['eligible_for_nested_group_cv'] for z in FEASIBILITY if z['regime']==reg and z['grouping']=='publication'):
        SPLITS[reg]=[];continue
    g=d.publication_id.astype(str).to_numpy()
    outer=list(LeaveOneGroupOut().split(d[FEATURES[reg]],d[TARGET],g))
    if MAX_OUTER is not None:outer=outer[:MAX_OUTER]
    SPLITS[reg]=outer
    for fold,(tr,te) in enumerate(outer,1):
        assert set(g[tr]).isdisjoint(set(g[te]))
        for part,idx in [('train',tr),('test',te)]:
            split_rows.extend([{'regime':reg,'grouping':'publication','outer_fold':fold,'partition':part,'orig_id':d.iloc[i].orig_id,'publication_id':g[i]} for i in idx])
save_csv(split_rows,'03_splits','publication_outer_split_manifest.csv')
print({r:len(s) for r,s in SPLITS.items()})
# Exact numerical input fingerprints appearing in more than one declared publication.
# Important sensitivity diagnostic; distinct replicates are NOT automatically mislabeled as leakage.
FINGERPRINT_OVERLAPS=[]
for reg,d in RAW.items():
    feats=FEATURES[reg]
    for _,part in d.groupby(feats,dropna=False):
        group_labels=sorted(set(part.publication_id.astype(str)))
        if len(group_labels)>1:
            for row in part.itertuples():
                FINGERPRINT_OVERLAPS.append({'regime':reg,'orig_id':str(row.orig_id),
                    'publication_id':str(row.publication_id),'other_publications':';'.join(x for x in group_labels if x!=str(row.publication_id)),
                    'flag':'EXACT_CROSS_GROUP_FEATURE_OVERLAP_UNVERIFIED_INDEPENDENCE'})
CROSS_GROUP_OVERLAPS=pd.DataFrame(FINGERPRINT_OVERLAPS,columns=['regime','orig_id','publication_id','other_publications','flag'])
save_csv(CROSS_GROUP_OVERLAPS,'03_splits','cross_publication_exact_feature_overlap.csv')
if not CROSS_GROUP_OVERLAPS.empty:
    status('03_splits','CROSS_GROUP_IDENTICAL_FEATURES','CAUTION',
           f'{len(CROSS_GROUP_OVERLAPS)} row(s) have identical inputs across declared source groups; report sensitivity and verify independence',len(CROSS_GROUP_OVERLAPS))


{'cryogenic': 5, 'room': 5, 'high': 0}


## Cell 25 — Optional independent external dataset and lab-input gates

An actual independent external benchmark may be attached as `external_experiments.csv`, with the same numeric features and measured `composite_tc_w_mk`, plus independent identity/evidence. `measurement_records.csv` may contain heat-flow geometry, instrument uncertainties and calibration standards. **Do not** treat original held-out CV rows or Maxwell-computed values as newly performed laboratory measurements. Missing optional files are documented, not filled in with synthetic data.

In [13]:
def optional_input(name):
    p=INPUT_DIR/name
    if p.exists():return p
    root=Path('/kaggle/input')
    found=list(root.rglob(name)) if root.exists() else []
    if len(found)>1: raise RuntimeError(f'Ambiguous optional input {name}: {found}; set TC_INPUT_DIR')
    return found[0] if len(found)==1 else None
EXTERNAL_PATH=optional_input('external_experiments.csv')
MEASUREMENT_PATH=optional_input('measurement_records.csv')
EXTERNAL_RAW=pd.read_csv(EXTERNAL_PATH) if EXTERNAL_PATH else pd.DataFrame()
MEASUREMENT_RAW=pd.read_csv(MEASUREMENT_PATH) if MEASUREMENT_PATH else pd.DataFrame()
external_flags=[]
if EXTERNAL_RAW.empty: status('08_external','E8_INDEPENDENT_EXPERIMENTS','NOT_ESTIMABLE','No independent external_experiments.csv supplied')
if MEASUREMENT_RAW.empty:status('10_uncertainty','E10_LAB_RECORDS','NOT_ESTIMABLE','No heat-flow and calibration measurements provided')
save_json({'external_file':str(EXTERNAL_PATH) if EXTERNAL_PATH else None,'external_rows':len(EXTERNAL_RAW),'measurement_file':str(MEASUREMENT_PATH) if MEASUREMENT_PATH else None,'measurement_rows':len(MEASUREMENT_RAW)},'03_splits','external_and_lab_input_status.json')

PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/03_splits/external_and_lab_input_status.json')

## Cell 27 — Regressor registry and scientifically comparable baselines

E1 compares **Extra Trees (ET)**, **Gradient Boosting (GB)**, **XGBoost**, **Random Forest (RF)**, and **SVR**. The additional **train-group-mean** and **matrix-conductivity** benchmarks protect against overstating modest model gains. XGBoost must be importable; otherwise the notebook emits an explicit missing-model status (not a disguised replacement). All models train on a CPU with controlled stochastic seeds.

In [14]:
try:
    import xgboost
    from xgboost import XGBRegressor
    XGB_READY=True
    run_info['xgboost_version']=xgboost.__version__
except Exception as exc:
    XGB_READY=False
    if RUN_MODE=='full': raise ImportError('XGBoost is required for five-model full Phase 2; install xgboost in Kaggle before rerunning') from exc
    status('04_models','E1_XGBOOST','NOT_ESTIMABLE',f'XGBoost import failed: {type(exc).__name__}: {exc}')
def model_base(name):
    if name=='ExtraTrees':return ExtraTreesRegressor(n_estimators=110,min_samples_leaf=1,random_state=SEED,n_jobs=CPU_JOBS)
    if name=='GradientBoosting':return GradientBoostingRegressor(n_estimators=100,random_state=SEED)
    if name=='XGBoost':return XGBRegressor(n_estimators=100,max_depth=3,learning_rate=0.06,subsample=1.0,colsample_bytree=1.0,reg_lambda=1.0,random_state=SEED,n_jobs=CPU_JOBS,objective='reg:squarederror',verbosity=0,tree_method='hist')
    if name=='RandomForest':return RandomForestRegressor(n_estimators=110,min_samples_leaf=1,random_state=SEED,n_jobs=CPU_JOBS)
    if name=='SVR':return SVR(kernel='rbf',C=2.0,epsilon=0.1,gamma='scale')
    if name=='GPR':return GaussianProcessRegressor(kernel=ConstantKernel(1.)*RBF(1.)+WhiteKernel(0.01),alpha=1e-4,optimizer=None,normalize_y=True,random_state=SEED)
    if name=='MLP':return MLPRegressor(hidden_layer_sizes=(16,16),solver='lbfgs',max_iter=280,alpha=1e-3,random_state=SEED)
    raise ValueError(name)
MODEL_NAMES=['ExtraTrees','GradientBoosting','RandomForest','SVR']+(['XGBoost'] if XGB_READY else [])
if RUN_MODE=='smoke': MODEL_NAMES=[m for m in MODEL_NAMES if m in ('ExtraTrees','SVR')]
EXTRA_BASELINE_NAMES=['GPR','MLP'] if EXTRA_BASELINES else []
model_inventory=[{'model':m,'available':True,'cpu':True} for m in MODEL_NAMES]
model_inventory.extend([{'model':m,'available':True,'cpu':True,'role':'optional_reviewer_requested_baseline'} for m in EXTRA_BASELINE_NAMES])
model_inventory.extend([{'model':'train_mean','available':True,'cpu':True},{'model':'matrix_k','available':True,'cpu':True}])
save_csv(model_inventory,'04_models','model_registry.csv')
print(MODEL_NAMES)

['ExtraTrees', 'GradientBoosting', 'RandomForest', 'SVR', 'XGBoost']


## Cell 29 — Nested hyperparameter search spaces, fixed seeds and scoring

Use `RandomizedSearchCV` with group-preserving inner folds and negative MAE for model selection. Outer holdout rows cannot affect the search. Parameter spaces stay intentionally small for 256 observations; switch `RUN_MODE=full` for reviewer calculations and retain its search budget in the run manifest. Save tuning distributions and best-parameter tables.

In [15]:
PARAMS={
 'ExtraTrees':{'estimator__model__n_estimators':[80,140,220],'estimator__model__max_features':[1.0,'sqrt'],'estimator__model__min_samples_leaf':[1,2,3],'estimator__model__max_depth':[None,6,12]},
 'GradientBoosting':{'estimator__model__n_estimators':[70,120,200],'estimator__model__max_depth':[2,3],'estimator__model__learning_rate':[0.03,0.07,0.12],'estimator__model__min_samples_leaf':[1,3]},
 'RandomForest':{'estimator__model__n_estimators':[80,140,220],'estimator__model__max_features':[1.0,'sqrt'],'estimator__model__min_samples_leaf':[1,2,3],'estimator__model__max_depth':[None,6,12]},
 'SVR':{'estimator__model__C':[0.5,2.0,10.0,40.0],'estimator__model__epsilon':[0.02,0.07,0.15],'estimator__model__gamma':['scale',0.01,0.1]},
 'GPR':{'estimator__model__alpha':[1e-5,1e-3,1e-1]},
 'MLP':{'estimator__model__alpha':[1e-4,1e-3,1e-2],'estimator__model__hidden_layer_sizes':[(16,),(16,16),(32,16)]},
 'XGBoost':{'estimator__model__n_estimators':[70,130,210],'estimator__model__max_depth':[2,3,4],'estimator__model__learning_rate':[0.03,0.07,0.12],'estimator__model__subsample':[0.8,1.0],'estimator__model__min_child_weight':[1,3]}
}
save_json({k:v for k,v in PARAMS.items() if k in MODEL_NAMES+EXTRA_BASELINE_NAMES},'04_models','hyperparameter_search_spaces.json')
print('Nested objective: inner-fold MAE; final report includes MAE, RMSE and R².')

Nested objective: inner-fold MAE; final report includes MAE, RMSE and R².


## Cell 31 — Train-only Gaussian augmentation: scientifically labelled sensitivity analysis

E2 compares **no augmentation**, input-only Gaussian perturbations at **1%**, and an input-only **3% historical-magnitude sensitivity**. Two fit-time copies yield 3× training rows. The original historical script also perturbed targets and other predictors; this safer sensitivity experiment intentionally does not reproduce those physically ungrounded changes. It perturbs only positive constituent conductivity inputs; target, temperature and filler loading are not changed. The target staying fixed under changed inputs is an *approximate robustness sensitivity*, **not** a new thermodynamically validated independent measurement. Synthetic examples exist only inside `.fit()` on an outer/inner training partition; never in validation or testing. No pre-split augmentation or augmentation of descendants of held-out observations.

In [16]:
class InputPerturbationRegressor(RegressorMixin,BaseEstimator):
    '''Training-only input perturbation; never edits y or held-out examples.'''
    def __init__(self,estimator=None,augmentation='none',noise_level=0.01,copies=1,random_state=42):
        self.estimator=estimator
        self.augmentation=augmentation
        self.noise_level=noise_level
        self.copies=copies
        self.random_state=random_state
    def fit(self,X,y):
        X=np.asarray(X,dtype=float);y=np.asarray(y,dtype=float)
        if X.ndim!=2 or y.ndim!=1 or len(X)!=len(y):raise ValueError('Bad X/y shape')
        if not np.isfinite(X).all() or not np.isfinite(y).all():raise ValueError('Nonfinite training data')
        if self.augmentation=='none':
            xx,yy=X.copy(),y.copy()
        elif self.augmentation in ('gaussian_1pct','gaussian_3pct_historical'):
            required_level=0.01 if self.augmentation=='gaussian_1pct' else 0.03
            if not np.isclose(float(self.noise_level),required_level):
                raise ValueError('Augmentation label/noise standard deviation mismatch')
            if not isinstance(self.copies,int) or self.copies<1:raise ValueError('copies must be positive integer')
            rng=np.random.default_rng(self.random_state)
            parts=[X.copy()];targets=[y.copy()]
            for _ in range(self.copies):
                perturb=X.copy()
                perturb[:,:2]=np.maximum(1e-12,perturb[:,:2]*(1+rng.normal(0,self.noise_level,size=perturb[:,:2].shape)))
                parts.append(perturb)
                targets.append(y.copy())
            xx=np.vstack(parts);yy=np.concatenate(targets)
        else:raise ValueError('Unsupported augmentation '+str(self.augmentation))
        self.estimator_=clone(self.estimator)
        self.estimator_.fit(xx,yy)
        self.n_original_=len(X);self.n_fit_=len(xx)
        return self
    def predict(self,X):return self.estimator_.predict(np.asarray(X,dtype=float))
AUGS=['none','gaussian_1pct']+(['gaussian_3pct_historical'] if INCLUDE_HISTORICAL_3PCT else [])
def model_factory(name,augmentation='none'):
    pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),
                   ('scale',StandardScaler() if name in ('SVR','GPR','MLP') else 'passthrough'),
                   ('model',model_base(name))])
    levels={'none':0.01,'gaussian_1pct':0.01,'gaussian_3pct_historical':0.03}
    if augmentation not in levels:raise ValueError(augmentation)
    return InputPerturbationRegressor(estimator=pipe,augmentation=augmentation,
            noise_level=levels[augmentation],copies=AUGMENTATION_COPIES,random_state=SEED)
save_csv([{'augmentation':a,'noise_std_relative':{'none':0,'gaussian_1pct':0.01,'gaussian_3pct_historical':0.03}[a],
           'perturbed_features':'base_tc_w_mk;filler_tc_w_mk' if a!='none' else '',
           'outer_test_augmented':False,'is_independent_measurement':False,
           'interpretation':'Historical notebook implementation used noise_level=0.03 despite 1Percent filename' if 'historical' in a else 'Input-only sensitivity'}
          for a in AUGS], '04_models','augmentation_ablation_design.csv')
# Synthetic in-memory smoke tests: X and y remain unchanged, only raw columns 0 and 1 perturbed.
x=np.array([[0.2,20.,7.,80.],[0.3,10.,12.,120.]]);y=np.array([0.5,0.8]);xx=x.copy();yy=y.copy()
for level in [a for a in AUGS if a!='none']:
    test=model_factory('ExtraTrees',level)
    fitted=clone(test).fit(x,y)
    assert np.array_equal(x,xx) and np.array_equal(y,yy)
    assert fitted.n_fit_==(AUGMENTATION_COPIES+1)*fitted.n_original_ and np.isfinite(fitted.predict(x)).all()
try:
    model_factory('ExtraTrees','gaussian_1pct').set_params(noise_level=0.03).fit(x,y)
    raise AssertionError('Mislabeled 3% noise failed to raise')
except ValueError: pass
print('Augmentation tests passed: 1%, historical 3%, no modified targets or held-out rows.')


Augmentation tests passed: 1%, historical 3%, no modified targets or held-out rows.


## Cell 33 — Baseline definitions and per-fold evaluation metrics

The training-set mean is a fitted constant baseline, whereas the unfilled matrix conductivity (`base_tc_w_mk`) is a source-neutral heuristic, not a perfect physical effective-medium predictor. MAE/RMSE are always computed on untouched original held-out records. R² needs at least two nonidentical targets; otherwise leave it `NaN`. Do not claim calibration from training data.

In [17]:
def safe_metrics(y_true,y_pred):
    yt=np.asarray(y_true,dtype=float);yp=np.asarray(y_pred,dtype=float)
    return {'n':len(yt),'mae':float(mean_absolute_error(yt,yp)),'rmse':float(np.sqrt(mean_squared_error(yt,yp))),'r2':float(r2_score(yt,yp)) if len(yt)>=2 and len(np.unique(yt))>1 else np.nan,'bias':float(np.mean(yp-yt)),'mape_pct':float(np.mean(np.abs((yp-yt)/yt))*100) if (yt>0).all() else np.nan}
def fold_baselines(d,tr,te,reg,fold):
    mean=float(d.iloc[tr][TARGET].mean());actual=d.iloc[te][TARGET].to_numpy()
    for model,pred in [('train_mean',np.full(len(te),mean)),('matrix_k',d.iloc[te]['base_tc_w_mk'].to_numpy())]:
        for i,k in enumerate(te):
            PRED_ROWS.append({'regime':reg,'grouping':'publication_id','outer_fold':fold,'publication_id':str(d.iloc[k].publication_id),'orig_id':str(d.iloc[k].orig_id),'model':model,'augmentation':'none','observed':float(actual[i]),'predicted':float(pred[i]),'source_assignment_status':str(d.iloc[k].source_assignment_status)})
        FOLD_ROWS.append({'regime':reg,'grouping':'publication_id','outer_fold':fold,'model':model,'augmentation':'none','heldout_sources':','.join(sorted(set(d.iloc[te].publication_id.astype(str)))),**safe_metrics(actual,pred),'best_params_json':'{}','inner_best_mae':np.nan})
PRED_ROWS=[];FOLD_ROWS=[];SEARCH_ROWS=[]
print('Baselines available: source-disjoint train-mean and matrix-k prediction.')

Baselines available: source-disjoint train-mean and matrix-k prediction.


## Cell 35 — Core nested-CV runner with strict group isolation and incremental writes

**Most important training cell.** For a train/test outer split, build inner `GroupKFold` only on the outer training indices; optimize MAE, fit the selected pipeline on all outer training rows, and evaluate exclusively on untouched outer held-out rows. Write fold predictions, metrics, tuning results and optionally every fitted outer model *immediately* so a partial Kaggle execution still preserves progress. Training-only augmentation is nested inside the estimator and therefore inside every inner/outer `.fit()`. Source IDs are never predictors.

In [18]:
def run_nested_one(reg,d,fold,tr,te,name,aug,group_col='publication_id',exp='E1_E2'):
    gtr=d.iloc[tr][group_col].astype(str).to_numpy()
    n_in=min(INNER_SPLITS,len(np.unique(gtr)))
    if n_in<2:
        status('05_nested_cv',exp+'_'+reg+'_'+name,'NOT_ESTIMABLE','Fewer than two independent training groups')
        return
    test_g=d.iloc[te][group_col].astype(str).to_numpy()
    assert set(gtr).isdisjoint(set(test_g))
    xtr=d.iloc[tr][FEATURES[reg]].to_numpy(dtype=float);ytr=d.iloc[tr][TARGET].to_numpy(dtype=float)
    xte=d.iloc[te][FEATURES[reg]].to_numpy(dtype=float);yte=d.iloc[te][TARGET].to_numpy(dtype=float)
    inner=list(GroupKFold(n_splits=n_in).split(xtr,ytr,gtr))
    for itr,ite in inner:assert set(gtr[itr]).isdisjoint(set(gtr[ite]))
    search=RandomizedSearchCV(model_factory(name,aug),PARAMS[name],n_iter=SEARCH_ITER,scoring='neg_mean_absolute_error',cv=inner,refit=True,random_state=SEED,n_jobs=1,return_train_score=False,error_score='raise')
    search.fit(xtr,ytr)
    pred=search.best_estimator_.predict(xte)
    if not np.isfinite(pred).all():raise ValueError('Non-finite model predictions')
    info={'regime':reg,'grouping':group_col,'outer_fold':fold,'model':name,'augmentation':aug,'heldout_sources':','.join(sorted(set(test_g))),**safe_metrics(yte,pred),'best_params_json':json.dumps(search.best_params_,sort_keys=True),'inner_best_mae':float(-search.best_score_)}
    FOLD_ROWS.append(info)
    SEARCH_ROWS.append({'regime':reg,'grouping':group_col,'outer_fold':fold,'model':name,'augmentation':aug,'best_inner_mae':float(-search.best_score_),'best_params':json.dumps(search.best_params_,sort_keys=True),'n_inner_folds':len(inner)})
    for i,k in enumerate(te):
        PRED_ROWS.append({'regime':reg,'grouping':group_col,'outer_fold':fold,'publication_id':str(d.iloc[k].publication_id),'orig_id':str(d.iloc[k].orig_id),'model':name,'augmentation':aug,'observed':float(yte[i]),'predicted':float(pred[i]),'source_assignment_status':str(d.iloc[k].source_assignment_status)})
    tag=f'{reg}__{group_col}__fold{fold:02d}__{name}__{aug}'
    save_csv(pd.DataFrame(search.cv_results_).drop(columns=['params'],errors='ignore'),'05_nested_cv',tag+'__inner_search.csv')
    if SAVE_OUTER_MODELS:
        model_path=OUT/'04_models'/(tag+'__outer_model.joblib')
        joblib.dump(search.best_estimator_,model_path,compress=2);artifact(model_path,'04_models')
    save_csv(FOLD_ROWS,'05_nested_cv','outer_fold_metrics_RUNNING.csv')
    save_csv(PRED_ROWS,'05_nested_cv','outer_predictions_RUNNING.csv')
    save_csv(SEARCH_ROWS,'05_nested_cv','inner_tuning_best_RUNNING.csv')
    return info
print('Nested group isolation implemented with explicit held-out groups and saved inner search tables.')

Nested group isolation implemented with explicit held-out groups and saved inner search tables.


## Cell 37 — Run E1 and E2: all five regressors × three augmentation settings (none, 1%, historical 3%)

This is the expensive full CPU cell. For cryogenic and room-temperature regimes with enough publication groups, execute all five regressors × no-augmentation/1%-Gaussian/historical-magnitude-3%-Gaussian × leave-one-publication-out folds. For the high-temperature regime, **skip publication-CV** because its 42 rows have only one declared source. Save `outer_fold_metrics.csv`, `outer_predictions.csv` and `inner_tuning_best.csv` after each experiment. Baseline predictions are evaluated on identical folds. A smoke run intentionally does not produce a reviewer-ready performance estimate.

In [19]:
for reg,d in RAW.items():
    if not SPLITS[reg]:
        status('05_nested_cv','E1_'+reg,'NOT_ESTIMABLE','Publication-grouped nested CV impossible; no random fallback',len(d))
        continue
    for fold,(tr,te) in enumerate(SPLITS[reg],1):
        fold_baselines(d,tr,te,reg,fold)
        for name in MODEL_NAMES:
            for aug in AUGS:
                t0=time.time()
                run_nested_one(reg,d,fold,tr,te,name,aug)
                print(f'{reg} fold={fold} {name}/{aug} completed in {time.time()-t0:.1f}s')
        for baseline_name in EXTRA_BASELINE_NAMES:
            run_nested_one(reg,d,fold,tr,te,baseline_name,'none',exp='E1_EXTRA_BASELINES')
    status('05_nested_cv','E1_E2_'+reg,'PROVISIONAL_DONE' if RUN_MODE=='full' else 'SMOKE_NONFINAL','Numerically grouped on unverified formerly-random placeholders; NOT a verified source-generalization result',len(d))
FOLD_DF=pd.DataFrame(FOLD_ROWS)
OOF_DF=pd.DataFrame(PRED_ROWS)
SEARCH_DF=pd.DataFrame(SEARCH_ROWS)
save_csv(FOLD_DF,'05_nested_cv','outer_fold_metrics.csv')
save_csv(OOF_DF,'05_nested_cv','outer_predictions.csv')
save_csv(SEARCH_DF,'05_nested_cv','inner_tuning_best.csv')
print('Prediction rows:',len(OOF_DF),'fold metric rows:',len(FOLD_DF))

cryogenic fold=1 ExtraTrees/none completed in 2.5s
cryogenic fold=1 ExtraTrees/gaussian_1pct completed in 2.5s
cryogenic fold=1 ExtraTrees/gaussian_3pct_historical completed in 2.5s
cryogenic fold=1 GradientBoosting/none completed in 1.4s
cryogenic fold=1 GradientBoosting/gaussian_1pct completed in 1.5s
cryogenic fold=1 GradientBoosting/gaussian_3pct_historical completed in 1.5s
cryogenic fold=1 RandomForest/none completed in 3.3s
cryogenic fold=1 RandomForest/gaussian_1pct completed in 3.4s
cryogenic fold=1 RandomForest/gaussian_3pct_historical completed in 3.3s
cryogenic fold=1 SVR/none completed in 0.1s
cryogenic fold=1 SVR/gaussian_1pct completed in 0.1s
cryogenic fold=1 SVR/gaussian_3pct_historical completed in 0.1s
cryogenic fold=1 XGBoost/none completed in 0.6s
cryogenic fold=1 XGBoost/gaussian_1pct completed in 0.5s
cryogenic fold=1 XGBoost/gaussian_3pct_historical completed in 0.5s


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _chec

cryogenic fold=2 ExtraTrees/none completed in 2.5s
cryogenic fold=2 ExtraTrees/gaussian_1pct completed in 2.6s
cryogenic fold=2 ExtraTrees/gaussian_3pct_historical completed in 2.7s
cryogenic fold=2 GradientBoosting/none completed in 1.4s
cryogenic fold=2 GradientBoosting/gaussian_1pct completed in 1.5s
cryogenic fold=2 GradientBoosting/gaussian_3pct_historical completed in 1.6s
cryogenic fold=2 RandomForest/none completed in 3.3s
cryogenic fold=2 RandomForest/gaussian_1pct completed in 3.5s
cryogenic fold=2 RandomForest/gaussian_3pct_historical completed in 3.4s
cryogenic fold=2 SVR/none completed in 0.1s
cryogenic fold=2 SVR/gaussian_1pct completed in 0.1s
cryogenic fold=2 SVR/gaussian_3pct_historical completed in 0.1s
cryogenic fold=2 XGBoost/none completed in 0.5s
cryogenic fold=2 XGBoost/gaussian_1pct completed in 0.5s
cryogenic fold=2 XGBoost/gaussian_3pct_historical completed in 0.5s


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _chec

cryogenic fold=3 ExtraTrees/none completed in 2.5s
cryogenic fold=3 ExtraTrees/gaussian_1pct completed in 2.6s
cryogenic fold=3 ExtraTrees/gaussian_3pct_historical completed in 2.5s
cryogenic fold=3 GradientBoosting/none completed in 1.4s
cryogenic fold=3 GradientBoosting/gaussian_1pct completed in 1.5s
cryogenic fold=3 GradientBoosting/gaussian_3pct_historical completed in 1.5s
cryogenic fold=3 RandomForest/none completed in 3.4s
cryogenic fold=3 RandomForest/gaussian_1pct completed in 3.4s
cryogenic fold=3 RandomForest/gaussian_3pct_historical completed in 3.4s
cryogenic fold=3 SVR/none completed in 0.1s
cryogenic fold=3 SVR/gaussian_1pct completed in 0.1s
cryogenic fold=3 SVR/gaussian_3pct_historical completed in 0.1s
cryogenic fold=3 XGBoost/none completed in 0.5s
cryogenic fold=3 XGBoost/gaussian_1pct completed in 0.6s
cryogenic fold=3 XGBoost/gaussian_3pct_historical completed in 0.5s


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _chec

... (12 outputs left out)

room fold=5 ExtraTrees/none completed in 2.5s
room fold=5 ExtraTrees/gaussian_1pct completed in 2.7s
room fold=5 ExtraTrees/gaussian_3pct_historical completed in 2.6s
room fold=5 GradientBoosting/none completed in 1.5s
room fold=5 GradientBoosting/gaussian_1pct completed in 2.0s
room fold=5 GradientBoosting/gaussian_3pct_historical completed in 1.9s
room fold=5 RandomForest/none completed in 3.5s
room fold=5 RandomForest/gaussian_1pct completed in 3.5s
room fold=5 RandomForest/gaussian_3pct_historical completed in 3.4s
room fold=5 SVR/none completed in 0.1s
room fold=5 SVR/gaussian_1pct completed in 0.2s
room fold=5 SVR/gaussian_3pct_historical completed in 0.2s
room fold=5 XGBoost/none completed in 0.6s
room fold=5 XGBoost/gaussian_1pct completed in 0.6s
room fold=5 XGBoost/gaussian_3pct_historical completed in 0.6s


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _chec

Prediction rows: 4066 fold metric rows: 190


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


## Cell 39 — E6 cross-regime OOF summaries, held-out source scores and plots

Aggregate all original out-of-fold predictions **by regime and model**. Distinguish *row-weighted OOF metrics* from *source-macro-average MAE*; do not average fold R² naively. A model's score is not treated as independent empirical testing merely because each observation was once withheld. Export performance ranking and held-out-source error tables, then plot per-regime comparisons.

In [20]:
SUMMARY=[];GROUP_SCORES=[]
if not OOF_DF.empty:
    for (reg,name,aug),g in OOF_DF.groupby(['regime','model','augmentation']):
        m=safe_metrics(g.observed,g.predicted)
        sg=g.assign(_abs_error=(g.observed-g.predicted).abs()).groupby('publication_id')['_abs_error'].mean()
        SUMMARY.append({'regime':reg,'model':name,'augmentation':aug,**m,'source_macro_mae':float(sg.mean()),'n_heldout_groups':int(len(sg)),'claim_status':'PROVISIONAL_AUTHOR_SOURCE_LABELS'})
        for pg,sval in sg.items():GROUP_SCORES.append({'regime':reg,'model':name,'augmentation':aug,'publication_id':pg,'group_mae':sval})
SCORES=pd.DataFrame(SUMMARY);GROUP_MAE=pd.DataFrame(GROUP_SCORES)
save_csv(SCORES,'05_nested_cv','oof_overall_metrics.csv');save_csv(GROUP_MAE,'05_nested_cv','per_publication_mae.csv')
for reg,g in SCORES[SCORES.augmentation.eq('none')].groupby('regime'):
    fig,ax=plt.subplots(figsize=(8,4));gg=g.sort_values('source_macro_mae')
    ax.barh(gg.model,gg.source_macro_mae);ax.set_xlabel('Held-out publication macro-MAE (W/m·K)');ax.set_title(reg+' | provisionally author-labelled publication holdout')
    save_figure('12_figures',f'{reg}_model_mae.png')
print(SCORES[['regime','model','augmentation','mae','rmse','r2','source_macro_mae']].round(4).to_string(index=False) if not SCORES.empty else 'No grouped OOF metrics estimable.')
SENSITIVITY=[]
if not OOF_DF.empty:
    for (reg,name,aug),part in OOF_DF[OOF_DF.grouping.eq('publication_id')].groupby(['regime','model','augmentation']):
        suspect=set(CROSS_GROUP_OVERLAPS.loc[CROSS_GROUP_OVERLAPS.regime.eq(reg),'orig_id'])
        clear=part[~part.orig_id.astype(str).isin(suspect)]
        SENSITIVITY.append({'regime':reg,'model':name,'augmentation':aug,
            'n_all':len(part),'n_without_cross_group_input_overlap':len(clear),
            'mae_without_cross_group_input_overlap':float(mean_absolute_error(clear.observed,clear.predicted)) if len(clear) else np.nan,
            'status':'PROVISIONAL_UNVERIFIED_GROUPS'})
save_csv(SENSITIVITY,'05_nested_cv','cross_publication_input_overlap_sensitivity.csv')


   regime            model             augmentation    mae   rmse      r2  source_macro_mae
cryogenic       ExtraTrees            gaussian_1pct 0.0860 0.1250  0.7604            0.0856
cryogenic       ExtraTrees gaussian_3pct_historical 0.0873 0.1277  0.7503            0.0869
cryogenic       ExtraTrees                     none 0.0885 0.1250  0.7607            0.0880
cryogenic              GPR                     none 0.0998 0.1482  0.6632            0.0990
cryogenic GradientBoosting            gaussian_1pct 0.0843 0.1215  0.7737            0.0841
cryogenic GradientBoosting gaussian_3pct_historical 0.0866 0.1275  0.7511            0.0864
cryogenic GradientBoosting                     none 0.0804 0.1171  0.7899            0.0801
cryogenic              MLP                     none 0.0986 0.1522  0.6449            0.0983
cryogenic     RandomForest            gaussian_1pct 0.0899 0.1210  0.7756            0.0892
cryogenic     RandomForest gaussian_3pct_historical 0.0861 0.1169  0.7905       

PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/05_nested_cv/cross_publication_input_overlap_sensitivity.csv')

## Cell 41 — E1 model comparisons using group-paired uncertainty and multiplicity

Compare models against ExtraTrees **within the same held-out publication groups**. With five groups, statistical power is limited; use exact sign-flip enumeration for group-mean MAE differences and source-resampled percentile confidence intervals. Apply Holm's adjustment within regime for several model comparisons. These p-values are exploratory, **not** proof of universal model superiority or independent source attribution.

In [21]:
COMPARISONS=[]
def exact_signflip(d):
    d=np.asarray(d,float)
    if len(d)<4:return np.nan
    if len(d)>16: # deterministic Monte Carlo, avoid exponential explosion
        rng=np.random.default_rng(SEED);signs=rng.choice([-1.,1.],size=(20000,len(d)))
        vals=np.abs((signs*d).mean(axis=1));obs=abs(d.mean())
        return float((1+sum(vals>=obs-1e-12))/(len(vals)+1))
    obs=abs(d.mean())
    vals=[abs(np.mean(d*np.array(s))) for s in itertools.product([-1.,1.],repeat=len(d))]
    return float(sum(v>=obs-1e-12 for v in vals)/len(vals))
def group_boot_ci(d,n=1200):
    if len(d)<3:return (np.nan,np.nan)
    g=np.random.default_rng(SEED);idx=g.integers(0,len(d),(n,len(d)));s=np.asarray(d)[idx].mean(axis=1)
    return (float(np.quantile(s,0.025)),float(np.quantile(s,0.975)))
if not GROUP_MAE.empty:
    for reg,g in GROUP_MAE[GROUP_MAE.augmentation.eq('none')].groupby('regime'):
        base=g[g.model.eq('ExtraTrees')][['publication_id','group_mae']].rename(columns={'group_mae':'baseline_mae'})
        for name in MODEL_NAMES:
            if name=='ExtraTrees':continue
            q=g[g.model.eq(name)][['publication_id','group_mae']].merge(base,on='publication_id')
            if len(q)<4: continue
            delta=(q.group_mae-q.baseline_mae).to_numpy();low,high=group_boot_ci(delta)
            COMPARISONS.append({'regime':reg,'comparison':name+' minus ExtraTrees','n_publications':len(q),'mean_delta_mae':float(delta.mean()),'ci95_low':low,'ci95_high':high,'p_exact_signflip':exact_signflip(delta),'p_holm':np.nan,'interpretation':'EXPLORATORY_ONLY: publication groups derive from previously randomized unverified labels'})
COMP=pd.DataFrame(COMPARISONS)
if not COMP.empty:
    for reg,ix in COMP.groupby('regime').groups.items():
        inds=list(ix);p=COMP.loc[inds,'p_exact_signflip'].to_numpy();order=np.argsort(p);adj=np.empty(len(p));prev=0
        for j,k in enumerate(order):prev=max(prev,float(min(1,(len(p)-j)*p[k])));adj[k]=prev
        COMP.loc[inds,'p_holm']=adj
save_csv(COMP,'06_stats','paired_group_model_comparisons.csv')
print('Paired model comparisons:',len(COMP))
RMSE_COMPARISONS=[]
if not FOLD_DF.empty:
    fp=FOLD_DF[(FOLD_DF.grouping.eq('publication_id')) & (FOLD_DF.augmentation.eq('none'))]
    for reg,part in fp.groupby('regime'):
        base=part[part.model.eq('ExtraTrees')][['outer_fold','rmse']].rename(columns={'rmse':'baseline_rmse'})
        for name in [x for x in MODEL_NAMES if x!='ExtraTrees']:
            curr=part[part.model.eq(name)][['outer_fold','rmse']]
            paired=curr.merge(base,on='outer_fold',validate='one_to_one')
            if len(paired)<4:continue
            diff=(paired.rmse-paired.baseline_rmse).to_numpy()
            lower,upper=group_boot_ci(diff)
            RMSE_COMPARISONS.append({'regime':reg,'comparison':name+' minus ExtraTrees',
                'n_publications':len(diff),'mean_delta_rmse':float(diff.mean()),
                'ci95_low':lower,'ci95_high':upper,'p_exact_signflip':exact_signflip(diff),
                'status':'PROVISIONAL_UNVERIFIED_SOURCE_LABELS'})
RMSE_COMP=pd.DataFrame(RMSE_COMPARISONS)
if not RMSE_COMP.empty:
    for reg,idx in RMSE_COMP.groupby('regime').groups.items():
        ids=list(idx);p=RMSE_COMP.loc[ids,'p_exact_signflip'].to_numpy();rank=np.argsort(p)
        last=0.;adjusted=np.empty(len(p))
        for i,k in enumerate(rank):
            last=max(last,min(1.,(len(p)-i)*p[k]));adjusted[k]=last
        RMSE_COMP.loc[ids,'p_holm']=adjusted
save_csv(RMSE_COMP,'06_stats','paired_group_rmse_comparisons.csv')


Paired model comparisons: 8


PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/06_stats/paired_group_rmse_comparisons.csv')

## Cell 43 — E2 paired augmentation ablation and train-copy counts

For each model, compare held-out-source MAE with `gaussian_1pct` versus `none` on identical publication splits. Report **augmentation minus no augmentation**, confidence interval and exact sign-flip p-value where enough source groups exist. Since no measured feature-error distribution is supplied, this is a computational perturbation sensitivity and must not be described as new physical samples.

In [22]:
AUG_STATS=[]
if not GROUP_MAE.empty:
    for (reg,name),g in GROUP_MAE[GROUP_MAE.model.isin(MODEL_NAMES)].groupby(['regime','model']):
        base=g[g.augmentation.eq('none')][['publication_id','group_mae']].rename(columns={'group_mae':'without_aug_mae'})
        for aug in [x for x in AUGS if x!='none']:
            altered=g[g.augmentation.eq(aug)][['publication_id','group_mae']].rename(columns={'group_mae':'with_aug_mae'})
            t=altered.merge(base,on='publication_id',validate='one_to_one')
            if t.empty:continue
            diff=(t.with_aug_mae-t.without_aug_mae).to_numpy();lo,hi=group_boot_ci(diff)
            AUG_STATS.append({'regime':reg,'model':name,'augmentation':aug,'n_publications':len(diff),
                'delta_mae_aug_minus_none':float(diff.mean()),'ci95_low':lo,'ci95_high':hi,
                'p_exact_signflip':exact_signflip(diff),'synthetic_training_copies_per_original':AUGMENTATION_COPIES,
                'scientific_status':'PROVISIONAL: raw-input sensitivity, NOT additional independent measurements'})
AUG_RESULTS=pd.DataFrame(AUG_STATS)
save_csv(AUG_RESULTS,'06_stats','augmentation_paired_ablation.csv')
if not AUG_RESULTS.empty:
    labels=AUG_RESULTS.regime+' '+AUG_RESULTS.model+' '+AUG_RESULTS.augmentation
    fig,ax=plt.subplots(figsize=(10,max(4,len(AUG_RESULTS)*0.34)))
    ax.barh(labels,AUG_RESULTS.delta_mae_aug_minus_none);ax.axvline(0,color='black',lw=0.8)
    ax.set_xlabel('Held-out source macro-MAE change; negative favours perturbation')
    save_figure('12_figures','augmentation_effect.png')
print('Paired augmentation evaluations:',len(AUG_RESULTS))


Paired augmentation evaluations: 20


## Cell 45 — E3 material-system grouped validation (strict fail-safe)

If **true per-row material-system identifiers** exist and at least three distinct groups are present, rerun the grouped nested design with `material_system_id` as the held-out label. Publication grouping alone is not an adequate substitute for this experiment. Without material IDs, emit an empty E3 result and a `NOT_ESTIMABLE` status; do not infer materials from conductivity thresholds.

In [23]:
MATERIAL_TEST=[]
for reg,d in RAW.items():
    eligible=next(f['eligible_for_nested_group_cv'] for f in FEASIBILITY if f['regime']==reg and f['grouping']=='material')
    if not eligible:
        status('06_stats',f'E3_MATERIAL_CV_{reg}','NOT_ESTIMABLE','Per-row material IDs absent or insufficient; publication ID not substituted',len(d))
        continue
    groups=d.material_system_id.astype(str).to_numpy()
    folds=list(LeaveOneGroupOut().split(d[FEATURES[reg]],d[TARGET],groups))
    if MAX_OUTER is not None:folds=folds[:MAX_OUTER]
    for fold,(tr,te) in enumerate(folds,1):
        result=run_nested_one(reg,d,fold,tr,te,'ExtraTrees','none',group_col='material_system_id',exp='E3')
        if result:MATERIAL_TEST.append(result)
save_csv(MATERIAL_TEST,'06_stats','material_grouped_nested_cv.csv')
print('Material-grouped outer folds completed:',len(MATERIAL_TEST))
# E3 predictions are separately recorded; primary E1/E2 OOF scores remain publication-only.
material_predictions=[z for z in PRED_ROWS if z['grouping']=='material_system_id']
save_csv(material_predictions,'06_stats','material_grouped_predictions.csv')


Material-grouped outer folds completed: 0


PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/06_stats/material_grouped_predictions.csv')

## Cell 47 — E4 strict epoxy-only versus heterogeneous training feasibility

Reviewers require distinguishing epoxy from fiberglass, graphite, copper and thermoplastics. **Do not** claim an epoxy-only benchmark based solely on the material studied by a cited paper. Strict epoxy eligibility requires actual row-level provenance, experimental status and verified formulation/loading. Run a separate nested comparison only if at least three independent *verified epoxy* sources remain; otherwise report that E4 is scientifically not estimable and retain study-level candidate counts as advisory.

In [24]:
EPOXY_CV=[]
epoxy_start_prediction_index=len(PRED_ROWS)
for reg,d in DATA.groupby('temperature_regime',sort=False):
    dd=d[d.strict_epoxy_eligible].copy().reset_index(drop=True)
    ng=dd.publication_id.nunique()
    if ng<3:
        status('06_stats',f'E4_STRICT_EPOXY_{reg}','NOT_ESTIMABLE',f'{len(dd)} strict verified rows across {ng} independent sources',len(dd))
        continue
    folds=list(LeaveOneGroupOut().split(dd[FEATURES[reg]],dd[TARGET],dd.publication_id))
    if MAX_OUTER is not None:folds=folds[:MAX_OUTER]
    for j,(tr,te) in enumerate(folds,1):
        result=run_nested_one(reg,dd,j,tr,te,'ExtraTrees','none',exp='E4_STRICT_EPOXY')
        if result:EPOXY_CV.append(result)
save_csv(EPOXY_CV,'06_stats','strict_epoxy_grouped_results.csv')
print('Strict-epoxy folds:',len(EPOXY_CV),'(zero is an evidence-driven result, not a missing execution)')
epoxy_predictions=PRED_ROWS[epoxy_start_prediction_index:]
# The core E1/E2 predictions may contain the same IDs; export only records recorded after E4 began.
# A deterministic dedicated experiment label is attached below for future strict-eligible cohorts.
save_csv(EPOXY_CV,'06_stats','strict_epoxy_nested_metrics.csv')
save_csv(epoxy_predictions,'06_stats','strict_epoxy_grouped_predictions.csv')


Strict-epoxy folds: 0 (zero is an evidence-driven result, not a missing execution)


PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/06_stats/strict_epoxy_grouped_predictions.csv')

## Cell 49 — Final models: all regimes, both ablations, machine-readable registry

After outer-CV evaluation, refit each requested regressor on each entire original regime for future application. For cryogenic/room, choose a final parameter configuration by frequency of inner-fold selections; **never** use the held-out group's target within its own outer tuning. For high-temperature rows, fit fixed-config **exploratory descriptive models** without claiming publication-grouped generalization. Save every model as a `joblib` file and a registry with training count, status, features, provenance condition and SHA.

In [25]:
FINAL_MODELS={};FINAL_REGISTRY=[]
for reg,d in RAW.items():
    X=d[FEATURES[reg]].to_numpy(dtype=float);y=d[TARGET].to_numpy(dtype=float)
    groups=d.publication_id.astype(str).to_numpy();ng=len(np.unique(groups))
    for name in MODEL_NAMES:
        for aug in AUGS:
            model=model_factory(name,aug)
            chosen={}
            if RUN_MODE=='full' and ng>=3:
                cv=list(GroupKFold(n_splits=min(INNER_SPLITS,ng)).split(X,y,groups))
                for a,b in cv:assert set(groups[a]).isdisjoint(set(groups[b]))
                tune=RandomizedSearchCV(model,PARAMS[name],n_iter=SEARCH_ITER,scoring='neg_mean_absolute_error',
                        cv=cv,refit=True,random_state=SEED,n_jobs=1,error_score='raise')
                tune.fit(X,y)
                chosen=tune.best_params_
                model=tune.best_estimator_
                selection='FULL_DATA_GROUP_CV_RESEARCH_FIT_NOT_UNSEEN_TEST'
                save_csv(pd.DataFrame(tune.cv_results_).drop(columns=['params'],errors='ignore'),
                        '04_models',f'FINAL_TUNING__{reg}__{name}__{aug}.csv')
            else:
                model.fit(X,y)
                selection=('SMOKE_FIXED_DEFAULT_NOT_FINAL' if RUN_MODE=='smoke' else 'SINGLE_SOURCE_FIXED_DEFAULT_NO_GROUP_CV')
            FINAL_MODELS[(reg,name,aug)]=model
            path=OUT/'04_models'/f'FINAL__{reg}__{name}__{aug}.joblib'
            joblib.dump(model,path,compress=3);artifact(path,'04_models')
            FINAL_REGISTRY.append({'regime':reg,'model':name,'augmentation':aug,'n_original_rows':len(d),
                 'n_train_fit_rows':model.n_fit_,'features':';'.join(FEATURES[reg]),
                 'publication_groups':ng,'validation_status':'UNVERIFIED_PUBLICATION_GROUPS_EXPLORATORY' if len(SPLITS[reg]) else 'NO_PUBLICATION_GROUPED_EVALUATION',
                 'parameter_selection':selection,'params_json':json.dumps(chosen,sort_keys=True),
                 'model_relative_path':path.relative_to(OUT).as_posix(),
                 'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
                 'model_source_population':'HETEROGENEOUS_LITERATURE_UNVERIFIED_RANDOM_ORIGIN_LABELS'})
    status('04_models',f'FINAL_FIT_{reg}','DONE_EXPLORATORY' if not SPLITS[reg] else 'DONE_GROUPED_PROVISIONAL',
           'Final fits on all available originals are NOT independent-test evidence',len(d))
FINAL_REGISTRY_DF=pd.DataFrame(FINAL_REGISTRY)
save_csv(FINAL_REGISTRY_DF,'04_models','final_model_registry.csv')
print('Saved final fitted model files:',len(FINAL_REGISTRY_DF))


Saved final fitted model files: 45


## Cell 51 — E1 feature interpretation: held-out permutation and tree impurity

Permutation importance is computed on **held-out publication rows** using each saved outer estimator, rather than on data that estimator trained on. This avoids in-sample inflation of permutation scores, though author-confirmed (not independently verified) group identities still constrain interpretation. Also export final-model tree impurity importance as a descriptive diagnostic. Feature importance is **associational, not causal**, and correlated predictors destabilize rankings.

In [26]:
PERM=[];GINI=[];PREFERRED={}
for reg,d in RAW.items():
    nonaug=SCORES[(SCORES.regime==reg)&(SCORES.augmentation=='none')&SCORES.model.isin(MODEL_NAMES)] if not SCORES.empty else pd.DataFrame()
    # Pre-specify ExtraTrees; selecting from the same heldout fold used for importance would bias it.
    name='ExtraTrees' if 'ExtraTrees' in MODEL_NAMES else MODEL_NAMES[0]
    if name not in MODEL_NAMES:name=MODEL_NAMES[0]
    PREFERRED[reg]=name
    final_model=FINAL_MODELS.get((reg,name,'none'))
    if final_model:
        regressor=final_model.estimator_.named_steps['model']
        if hasattr(regressor,'feature_importances_'):
            for c,v in zip(FEATURES[reg],regressor.feature_importances_):GINI.append({'regime':reg,'model':name,'feature':c,'train_impurity_importance':float(v),'status':'IN_SAMPLE_DESCRIPTIVE_NONCAUSAL_UNVERIFIED_LABELS'})
    if not SPLITS[reg]:continue
    for fold,(tr,te) in enumerate(SPLITS[reg],1):
        p=OUT/'04_models'/f'{reg}__publication_id__fold{fold:02d}__{name}__none__outer_model.joblib'
        if p.exists(): estimator=joblib.load(p)
        else:
            qs=SEARCH_DF[(SEARCH_DF.regime==reg)&(SEARCH_DF.outer_fold==fold)&(SEARCH_DF.model==name)&(SEARCH_DF.augmentation=='none')]
            estimator=model_factory(name,'none')
            if not qs.empty:estimator.set_params(**json.loads(qs.iloc[0].best_params))
            estimator.fit(d.iloc[tr][FEATURES[reg]],d.iloc[tr][TARGET])
        perm=permutation_importance(estimator,d.iloc[te][FEATURES[reg]].to_numpy(),d.iloc[te][TARGET].to_numpy(),scoring='neg_mean_absolute_error',n_repeats=10,random_state=SEED,n_jobs=1)
        for c,imp,std in zip(FEATURES[reg],perm.importances_mean,perm.importances_std):
            PERM.append({'regime':reg,'outer_fold':fold,'model':name,'feature':c,'permutation_delta_neg_MAE':float(imp),'permutation_sd':float(std),'heldout_publications':','.join(sorted(set(d.iloc[te].publication_id.astype(str))))})
save_csv(PERM,'07_interpretation','heldout_group_permutation_importance.csv')
save_csv(GINI,'07_interpretation','training_tree_feature_importance.csv')
save_json(PREFERRED,'07_interpretation','selected_model_for_diagnostics.json')
print('Chosen descriptive models:',PREFERRED)

Chosen descriptive models: {'cryogenic': 'ExtraTrees', 'room': 'ExtraTrees', 'high': 'ExtraTrees'}


## Cell 53 — SHAP for an available tree model with non-causal interpretation

Use SHAP, if installed, to explain the *fitted final tree model* on at most 50 original observations. This is a **descriptive within-dataset** explanation; because the final model is trained on all originals, it is not an independent accuracy estimate. If SHAP or a compatible tree explainer is unavailable, create a status row and continue without aborting reviewer-essential outputs. Save summary plot and mean-absolute contribution CSV where supported.

In [27]:
SHAP_OUTPUT=[]
try:
    import shap
    shap_version=getattr(shap,'__version__','unknown')
    if not RUN_SHAP:raise RuntimeError('SHAP disabled in smoke mode; full mode enables it')
    for reg,d in RAW.items():
        treename=PREFERRED.get(reg,'ExtraTrees')
        if treename not in ('ExtraTrees','GradientBoosting','RandomForest','XGBoost'):
            treename='ExtraTrees'
        if (reg,treename,'none') not in FINAL_MODELS:continue
        wrapper=FINAL_MODELS[(reg,treename,'none')]
        p=wrapper.estimator_
        inputs=d[FEATURES[reg]].to_numpy(float)[:min(50,len(d))]
        scaled=p[:-1].transform(inputs)
        expl=shap.TreeExplainer(p.named_steps['model'])
        s=expl.shap_values(scaled)
        a=np.asarray(s)
        if a.ndim!=2 or a.shape[1]!=len(FEATURES[reg]):raise RuntimeError(f'Unexpected SHAP value shape {a.shape}')
        for col,v in zip(FEATURES[reg],np.abs(a).mean(axis=0)):
            SHAP_OUTPUT.append({'regime':reg,'model':treename,'feature':col,'mean_abs_shap':float(v),'n_explained':len(inputs),'interpretation':'ASSOCIATION_NONCAUSAL'})
        shap.summary_plot(a,scaled,feature_names=FEATURES[reg],show=False,plot_type='bar')
        save_figure('12_figures',f'shap_summary_{reg}.png')
        status('07_interpretation','SHAP_'+reg,'DONE_DESCRIPTIVE',f'{len(inputs)} original records explained with {treename} using SHAP {shap_version}',len(inputs))
except Exception as exc:
    status('07_interpretation','SHAP_EXECUTION','NOT_ESTIMABLE',f'{type(exc).__name__}: {exc}')
save_csv(SHAP_OUTPUT,'07_interpretation','mean_absolute_shap.csv')
print('SHAP explanation rows:',len(SHAP_OUTPUT))

/tmp/ipykernel_17/2187710013.py:21: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(a,scaled,feature_names=FEATURES[reg],show=False,plot_type='bar')
/tmp/ipykernel_17/2187710013.py:21: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(a,scaled,feature_names=FEATURES[reg],show=False,plot_type='bar')
/tmp/ipykernel_17/2187710013.py:21: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(a,scaled,feature_names=FEATURE

SHAP explanation rows: 11


## Cell 55 — E8 independent external prediction and provenance/leakage tests

If `external_experiments.csv` exists, require its original ID and source identity to **not overlap** the training originals, and reject identical feature/target fingerprints. A source shared with training is *not* publication-independent. Only label external scores **independent** if source identity and experimental type are verifiable; otherwise keep predictions but mark errors as non-independent/provisional. This cannot be replaced by the training CSVs or theoretical Maxwell values.

In [28]:
EXT_PRED=[];EXT_SCORE=[];EXT_GATE=[]
if not EXTERNAL_RAW.empty:
    required=['temperature_regime','base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded','composite_tc_w_mk',
              'orig_id','publication_id','source_measurement_type','source_row_evidence']
    missing=[c for c in required if c not in EXTERNAL_RAW.columns]
    if missing:
        status('08_external','E8_EXTERNAL','NOT_ESTIMABLE','Missing required metadata: '+','.join(missing))
    else:
        for _,r in EXTERNAL_RAW.iterrows():
            reg=str(r.temperature_regime).strip().lower();oid=str(r.orig_id).strip()
            if reg not in FEATURES:
                EXT_GATE.append({'external_orig_id':oid,'eligible':False,'reason':'UNRECOGNIZED_REGIME'});continue
            required_numeric=FEATURES[reg]+[TARGET]
            numeric=[pd.to_numeric(r.get(x,np.nan),errors='coerce') for x in required_numeric]
            finite=all(np.isfinite(v) for v in numeric) and float(numeric[-1])>0
            if not finite:
                EXT_GATE.append({'external_orig_id':oid,'eligible':False,'reason':'INVALID_FEATURE_OR_TARGET'});continue
            vec=np.array(numeric[:-1],float);actual=float(numeric[-1]);main=RAW[reg]
            duplicate_input=bool(np.any(np.all(np.isclose(main[FEATURES[reg]].to_numpy(float),vec,rtol=1e-12,atol=1e-12),axis=1)))
            pid=str(r.publication_id).strip();source_row=str(r.source_row_evidence).strip()
            identity_ok=oid and oid.lower()!='nan' and pid and pid.lower()!='nan' and source_row and source_row.lower()!='nan'
            measured=str(r.source_measurement_type).lower().strip() in ['experimental','measured','physical_measurement']
            overlap=(oid in set(DATA.orig_id.astype(str))) or (pid in set(DATA.publication_id.astype(str))) or duplicate_input
            # Cannot independently validate an external paper/lab from its self-reported CSV alone.
            user_attested=identity_ok and measured and not overlap
            reason='USER_ATTESTED_CANDIDATE_NOT_INDEPENDENTLY_VERIFIED' if user_attested else 'MISSING_EVIDENCE_OR_OVERLAP'
            EXT_GATE.append({'external_orig_id':oid,'eligible':user_attested,'reason':reason})
            name=PREFERRED[reg]
            pred=float(FINAL_MODELS[(reg,name,'none')].predict(vec.reshape(1,-1))[0])
            EXT_PRED.append({'external_orig_id':oid,'regime':reg,'publication_id':pid,'observed':actual,
                'predicted':pred,'error':pred-actual,'abs_error':abs(pred-actual),
                'relative_abs_error_pct':100*abs(pred-actual)/actual,
                'independent_verified':False,'external_user_attested':user_attested,'model':name,'reason':reason})
        candidate=pd.DataFrame(EXT_PRED)
        if not candidate.empty and candidate.external_user_attested.any():
            for reg,part in candidate[candidate.external_user_attested].groupby('regime'):
                EXT_SCORE.append({'regime':reg,**safe_metrics(part.observed,part.predicted),
                   'n_verified_external':0,'n_user_attested_not_independently_verified':len(part),
                   'model':part.model.iloc[0], 'scientific_status':'PROVISIONAL_EXTERNAL_CANDIDATE'})
        status('08_external','E8_EXTERNAL','NOT_VERIFIED' if EXT_SCORE else 'NOT_ESTIMABLE',
          'CSV contents alone cannot establish independent experimental provenance; external scores remain provisional',len(EXT_PRED))
save_csv(EXT_GATE,'08_external','external_independence_gate.csv')
save_csv(EXT_PRED,'08_external','external_predictions.csv')
save_csv(EXT_SCORE,'08_external','verified_external_metrics.csv')
print('Independently externally verified experimental records: 0 (metadata-only gate is not external verification)')


Independently externally verified experimental records: 0 (metadata-only gate is not external verification)


## Cell 57 — E8 numerical OOD and range-extrapolation diagnostics

Estimate *numerical* out-of-domain status using **only the relevant outer training set** for comparison: flag if any held-out feature lies outside observed training min–max, and compute a standardized Euclidean distance to the training centroid. Feature scales are fit from training data only. These are simple extrapolation diagnostics, **not** proof of polymer chemistry novelty or aerospace qualification. Plot held-out absolute error against distance.

In [29]:
OOD=[]
if not OOF_DF.empty:
    for reg,d in RAW.items():
        if not SPLITS[reg]:continue
        subset=OOF_DF[(OOF_DF.regime==reg)&(OOF_DF.model==PREFERRED[reg])&(OOF_DF.augmentation=='none')&(OOF_DF.grouping=='publication_id')]
        for fold,(tr,te) in enumerate(SPLITS[reg],1):
            tx=d.iloc[tr][FEATURES[reg]].to_numpy(float);vx=d.iloc[te][FEATURES[reg]].to_numpy(float)
            mu=tx.mean(axis=0);std=tx.std(axis=0,ddof=0);std=np.where(std>0,std,1)
            dist=np.sqrt(np.sum(((vx-mu)/std)**2,axis=1))
            extrap=np.any((vx<tx.min(axis=0))|(vx>tx.max(axis=0)),axis=1)
            predmap=subset[subset.outer_fold.eq(fold)].set_index('orig_id').predicted.to_dict()
            for j,ix in enumerate(te):
                row=d.iloc[ix]; oid=str(row.orig_id)
                if oid in predmap:
                    OOD.append({'orig_id':oid,'regime':reg,'outer_fold':fold,'publication_id':str(row.publication_id),'numerical_centroid_distance':float(dist[j]),'feature_outside_outer_train_range':bool(extrap[j]),'abs_prediction_error':float(abs(predmap[oid]-row[TARGET])),'model':PREFERRED[reg]})
OOD_DF=pd.DataFrame(OOD)
save_csv(OOD_DF,'08_external','heldout_numerical_ood.csv')
if not OOD_DF.empty:
    for reg,d in OOD_DF.groupby('regime'):
        fig,ax=plt.subplots(figsize=(6,4));ax.scatter(d.numerical_centroid_distance,d.abs_prediction_error,alpha=.75);ax.set_xlabel('Training-normalized distance');ax.set_ylabel('Absolute prediction error (W/m·K)');ax.set_title(reg+' — numerical OOD only');save_figure('12_figures',f'ood_error_{reg}.png')
print('Numerical OOD diagnostics:',len(OOD_DF))

Numerical OOD diagnostics: 214


## Cell 59 — E5/E9 verified physics: Maxwell–Eucken, Bruggeman and HS bounds

Implement two-phase, isotropic, positive-conductivity estimates: (i) Maxwell spherical inclusions, (ii) symmetric Bruggeman implicit effective-medium root, and (iii) Hashin–Shtrikman (HS) upper/lower bounds for bulk isotropic components. **Requires verified volume fraction and compatible geometry**; anisotropic laminates, tubes, percolating morphologies and unknown units may invalidate comparison. Assert endpoint identities and that Maxwell/Bruggeman lie within basic constituent min/max for ordinary positive inputs; report theoretical bounds as conditional models, not universal experimental laws.

In [30]:
def maxwell_eucken(km,kf,phi):
    return km*((kf+2*km+2*phi*(kf-km))/(kf+2*km-phi*(kf-km)))
def bruggeman(km,kf,phi):
    if phi<=0:return km
    if phi>=1:return kf
    if math.isclose(km,kf):return km
    fun=lambda ke:(1-phi)*(km-ke)/(km+2*ke)+phi*(kf-ke)/(kf+2*ke)
    return brentq(fun,min(km,kf),max(km,kf),xtol=1e-12)
def hs_bounds(km,kf,phi):
    if math.isclose(km,kf):return km,km
    kl,kh=min(km,kf),max(km,kf)
    f_high=phi if kf>km else 1-phi
    f_low=1-f_high
    if f_high<=0:return kl,kl
    if f_low<=0:return kh,kh
    lower=kl+f_high/(1/(kh-kl)+f_low/(3*kl))
    upper=kh+f_low/(1/(kl-kh)+f_high/(3*kh))
    return float(lower),float(upper)
PHYS_TEST=[]
for km,kf in [(0.2,100),(100,0.2),(0.2,0.2)]:
    for ph in (0,0.2,0.5,0.9,1):
        mv=maxwell_eucken(km,kf,ph);bv=bruggeman(km,kf,ph);lo,hi=hs_bounds(km,kf,ph)
        ok=all(np.isfinite([mv,bv,lo,hi])) and lo<=hi+1e-10 and min(km,kf)-1e-8<=mv<=max(km,kf)+1e-8 and min(km,kf)-1e-8<=bv<=max(km,kf)+1e-8
        if ph==0:ok=ok and abs(mv-km)<1e-9 and abs(bv-km)<1e-9
        if ph==1:ok=ok and abs(mv-kf)<1e-9 and abs(bv-kf)<1e-9
        PHYS_TEST.append({'matrix_k':km,'filler_k':kf,'phi':ph,'maxwell':mv,'bruggeman':bv,'hs_lower':lo,'hs_upper':hi,'self_test_pass':bool(ok)})
save_csv(PHYS_TEST,'09_physics','effective_medium_unit_tests.csv')
assert all(r['self_test_pass'] for r in PHYS_TEST)
print('Theoretical model endpoint and interval self-tests passed:',len(PHYS_TEST))

Theoretical model endpoint and interval self-tests passed: 15


## Cell 61 — E5/E9 apply physical bounds only to eligible verified experiments

Compute models only for rows with a documented filler volume fraction, positive constituent conductivities, physically compatible material geometry, documented temperature conditions and measurement type. If none qualify, report `NOT_ESTIMABLE` and keep an empty structured CSV. Never convert weight% using guessed density or assume a nanotube/laminate geometry is spherical. Source evidence and material compatibility stay visible with each comparison. Plot physical prediction versus target only if evidence passes.

In [31]:
PHYS_ROWS=[]
for row in DATA.to_dict('records'):
    phi=row.get('filler_volume_fraction')
    if not np.isfinite(phi) or not (0<=phi<=1):continue
    if not str(row.get('source_row_evidence','')).strip():continue
    if str(row.get('source_measurement_type','')).lower() not in ['experimental','measured','physical_measurement']:continue
    # Strict spherical-isotropic eligibility requires explicit per-row geometry label; don't infer it.
    if str(row.get('morphology_verified','')).strip().lower() not in ['spherical_isotropic','isotropic_spherical_particles']:continue
    km=float(row['base_tc_w_mk']);kf=float(row['filler_tc_w_mk']);v=float(phi);act=float(row[TARGET]);lo,hi=hs_bounds(km,kf,v)
    PHYS_ROWS.append({'orig_id':row['orig_id'],'regime':row['temperature_regime'],'publication_id':row['publication_id'],'volume_fraction':v,'observed_composite_k':act,'maxwell_k':maxwell_eucken(km,kf,v),'bruggeman_k':bruggeman(km,kf,v),'hs_lower':lo,'hs_upper':hi,'outside_hs_bounds':bool(act<lo or act>hi),'geometry_evidence':'user_supplied_explicit_spherical_isotropic'})
PHYS_DF=pd.DataFrame(PHYS_ROWS,columns=['orig_id','regime','publication_id','volume_fraction','observed_composite_k','maxwell_k','bruggeman_k','hs_lower','hs_upper','outside_hs_bounds','geometry_evidence'])
save_csv(PHYS_DF,'09_physics','physics_applicable_comparisons.csv')
if PHYS_DF.empty:
    status('09_physics','E5_E9_EFFECTIVE_MEDIUM','NOT_ESTIMABLE','No experimental record has verified filler fraction plus explicit compatible particle morphology',0)
else:
    for reg,z in PHYS_DF.groupby('regime'):
        fig,ax=plt.subplots(figsize=(5.5,4.5));ax.scatter(z.observed_composite_k,z.maxwell_k,label='Maxwell');ax.scatter(z.observed_composite_k,z.bruggeman_k,label='Bruggeman');ax.plot([z.observed_composite_k.min(),z.observed_composite_k.max()],[z.observed_composite_k.min(),z.observed_composite_k.max()],'k--',lw=1);ax.set_xlabel('Observed composite k');ax.set_ylabel('Physical model k');ax.legend();ax.set_title(reg+' verified effective-medium comparison');save_figure('12_figures',f'physics_models_{reg}.png')
    status('09_physics','E5_E9_EFFECTIVE_MEDIUM','DONE_EVIDENCE_GATED','These analyses require morphological isotropy verified per row',len(PHYS_DF))
print('Physics-comparable experimental rows:',len(PHYS_DF))
# Fair same-observation comparison: only use source-held-out original ML predictions;
# never substitute final fitted training predictions for true held-out benchmarks.
SAME_ROW=[]
if not PHYS_DF.empty and not OOF_DF.empty:
    preds=OOF_DF[(OOF_DF.model.eq('ExtraTrees')) & (OOF_DF.augmentation.eq('none')) &
                 (OOF_DF.grouping.eq('publication_id'))][['regime','orig_id','observed','predicted']]
    joined=PHYS_DF.merge(preds,on=['regime','orig_id'],how='inner',validate='one_to_one')
    for rr in joined.itertuples():
        SAME_ROW.append({'regime':rr.regime,'orig_id':rr.orig_id,'publication_id':rr.publication_id,
            'observed_k':rr.observed,'ET_outer_holdout_k':rr.predicted,
            'maxwell_k':rr.maxwell_k,'bruggeman_k':rr.bruggeman_k,
            'ET_abs_error':abs(rr.predicted-rr.observed),
            'Maxwell_abs_error':abs(rr.maxwell_k-rr.observed),
            'Bruggeman_abs_error':abs(rr.bruggeman_k-rr.observed),
            'validation_status':'EXPLORATORY_UNVERIFIED_PUBLICATION_LABELS'})
save_csv(SAME_ROW,'09_physics','same_row_ml_vs_effective_medium.csv')


Physics-comparable experimental rows: 0


PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/09_physics/same_row_ml_vs_effective_medium.csv')

## Cell 63 — E7 group-disjoint split-conformal prediction intervals and coverage

Construct a **source-disjoint three-way split** for each eligible regime: **training publications**, a different **calibration publication**, and a different **test publication**. Choose the *predeclared fixed ExtraTrees model*, not the model chosen using all OOF test outcomes. Calibrate residual quantiles on the calibration source only, test interval coverage on the unseen source and output per-group coverage/widths and prediction CSV. This offers an empirical coverage audit, **not distribution-free guarantees under non-exchangeable publications**. If a finite 90% conformal quantile cannot be computed from the available calibration observations, mark the interval `NOT_ESTIMABLE` rather than invent one.

In [32]:
CONF_ROWS=[];CONF_STATS=[]
for reg,d in RAW.items():
    groups=sorted(d.publication_id.astype(str).unique().tolist())
    if len(groups)<3:
        status('10_uncertainty','E7_CONFORMAL_'+reg,'NOT_ESTIMABLE','Fewer than 3 publication groups; cannot make source-disjoint train/calibration/test')
        continue
    tests=groups if RUN_MODE=='full' else groups[:2]
    Xall=d[FEATURES[reg]].to_numpy(float);yall=d[TARGET].to_numpy(float)
    for test_group in tests:
        i=groups.index(test_group);cal_group=groups[(i+1)%len(groups)]
        train_groups=[x for x in groups if x not in (test_group,cal_group)]
        tr=d.publication_id.isin(train_groups).to_numpy();ca=d.publication_id.eq(cal_group).to_numpy();te=d.publication_id.eq(test_group).to_numpy()
        assert tr.sum()>0 and ca.sum()>0 and te.sum()>0
        if not (set(d.loc[tr,'publication_id']) & set(d.loc[ca,'publication_id']) or set(d.loc[tr,'publication_id']) & set(d.loc[te,'publication_id'])):pass
        else: raise AssertionError('Group leakage in conformal split')
        fixed=model_factory('ExtraTrees','none');fixed.fit(Xall[tr],yall[tr])
        cal_err=np.abs(yall[ca]-fixed.predict(Xall[ca]));n=len(cal_err)
        qrank=math.ceil((n+1)*(1-CONFORMAL_ALPHA))/n
        if qrank>1:
            CONF_STATS.append({'regime':reg,'test_publication':test_group,'cal_publication':cal_group,'n_cal':n,'n_test':int(te.sum()),'status':'NOT_ESTIMABLE_FINITE_90PCT_QUANTILE','coverage':np.nan,'mean_width':np.nan})
            continue
        q=float(np.quantile(cal_err,min(qrank,1),method='higher'))
        p=fixed.predict(Xall[te]);yy=yall[te];lower=p-q;upper=p+q
        inside=(yy>=lower)&(yy<=upper)
        for j,k in enumerate(np.flatnonzero(te)):
            CONF_ROWS.append({'regime':reg,'orig_id':str(d.iloc[k].orig_id),'test_publication':test_group,'cal_publication':cal_group,'n_calibration':n,'predicted':float(p[j]),'observed':float(yy[j]),'lower_90pct':float(lower[j]),'upper_90pct':float(upper[j]),'covered':bool(inside[j]),'width':float(2*q),'claimed_guarantee':'NONE_UNDER_PUBLICATION_SHIFT'})
        CONF_STATS.append({'regime':reg,'test_publication':test_group,'cal_publication':cal_group,'n_cal':n,'n_test':int(te.sum()),'status':'EMPIRICAL_SOURCE_DISJOINT','coverage':float(inside.mean()),'mean_width':float(2*q)})
CONF_DF=pd.DataFrame(CONF_ROWS);CONF_STATUS=pd.DataFrame(CONF_STATS)
save_csv(CONF_DF,'10_uncertainty','heldout_group_conformal_predictions.csv')
save_csv(CONF_STATUS,'10_uncertainty','heldout_group_interval_coverage.csv')
for reg,d in CONF_STATUS.groupby('regime') if not CONF_STATUS.empty else []:
    z=d[d.status.eq('EMPIRICAL_SOURCE_DISJOINT')]
    if not z.empty:
        fig,ax=plt.subplots(figsize=(8,3.7));ax.bar(z.test_publication,z.coverage);ax.axhline(1-CONFORMAL_ALPHA,ls='--',color='black',label='Nominal 90%');ax.set_ylim(0,1.05);ax.set_ylabel('Empirical held-out source coverage');ax.set_title(reg+' — source-disjoint conformal; no distribution-shift guarantee');ax.legend();save_figure('12_figures',f'conformal_coverage_{reg}.png')
print('Conformal outer records:',len(CONF_DF))

Conformal outer records: 214


## Cell 65 — E10 Fourier law, uncertainty propagation and instrument calibration gates

Optional experimental input `measurement_records.csv` must contain the actual measured heat flow `Q_W`, thickness `L_m`, heat-transfer cross-section `A_m2`, and temperature gradient `dT_K` (strictly positive) so Fourier's law gives **k = Q·L/(A·ΔT)** in W/(m·K). First-order independent-error propagation requires measured standard uncertainties for all four quantities (`Q_W_sigma`, `L_m_sigma`, `A_m2_sigma`, `dT_K_sigma`); otherwise output k but leave its uncertainty `NaN`. Important nonidealities (radiation, contact resistance, parasitic losses, calibration range, steady state and anisotropy) require separate evidence. Never fabricate calibration standards or claim uncertainty is fully traceable from an algebraic propagation alone.

In [33]:
FOURIER=[];CALIBRATION=[]
req=['Q_W','L_m','A_m2','dT_K']
sigs=['Q_W_sigma','L_m_sigma','A_m2_sigma','dT_K_sigma']
if MEASUREMENT_RAW.empty:
    status('10_uncertainty','E10_FOURIER','NOT_ESTIMABLE','No actual instrument geometry/heat-flow records')
else:
    missing=[c for c in req if c not in MEASUREMENT_RAW]
    if missing:status('10_uncertainty','E10_FOURIER','NOT_ESTIMABLE','Missing required measured quantities '+','.join(missing))
    else:
        for ix,rr in MEASUREMENT_RAW.iterrows():
            values=[pd.to_numeric(rr[x],errors='coerce') for x in req]
            if not all(np.isfinite(values)) or not all(v>0 for v in values):
                FOURIER.append({'measurement_id':str(rr.get('measurement_id',ix)),'status':'INVALID_MEASURED_VALUES','calculated_k_w_mk':np.nan,'u_k_w_mk':np.nan});continue
            Q,L,A,T=values;k=Q*L/(A*T)
            sigma=np.nan
            if all(s in MEASUREMENT_RAW for s in sigs):
                u=[pd.to_numeric(rr[s],errors='coerce') for s in sigs]
                if all(np.isfinite(u)) and all(v>=0 for v in u):sigma=float(k*np.sqrt(sum((a/b)**2 for a,b in zip(u,values))))
            FOURIER.append({'measurement_id':str(rr.get('measurement_id',ix)),'status':'FOURIER_ONLY_UNCERTAINTY_PARTIAL' if np.isfinite(sigma) else 'FOURIER_ONLY_UNCERTAINTY_NOT_ESTIMABLE','calculated_k_w_mk':float(k),'u_k_w_mk':sigma,'sample_id':str(rr.get('sample_id','')),'calibration_traceability':str(rr.get('calibration_document',''))})
        if 'certified_reference_k_w_mk' in MEASUREMENT_RAW:
            for ix,rr in MEASUREMENT_RAW.iterrows():
                cert=pd.to_numeric(rr.certified_reference_k_w_mk,errors='coerce')
                match=next((f for f in FOURIER if f['measurement_id']==str(rr.get('measurement_id',ix))),None)
                if np.isfinite(cert) and cert>0 and match and np.isfinite(match['calculated_k_w_mk']):
                    CALIBRATION.append({'measurement_id':match['measurement_id'],'certified_reference_k_w_mk':float(cert),'measured_k_w_mk':match['calculated_k_w_mk'],'bias_w_mk':match['calculated_k_w_mk']-float(cert),'calibration_document_present':bool(str(rr.get('calibration_document','')).strip())})
        status('10_uncertainty','E10_FOURIER','DONE_PARTIAL' if FOURIER else 'NOT_ESTIMABLE','A Fourier calculation is not a full thermal-conductivity calibration or GUM uncertainty budget',len(FOURIER))
if not CALIBRATION:status('10_uncertainty','E10_CALIBRATION','NOT_ESTIMABLE','No compatible certified calibration standard/result with traceability supplied')
save_csv(FOURIER,'10_uncertainty','fourier_law_measurements.csv')
save_csv(CALIBRATION,'10_uncertainty','standard_reference_calibration_checks.csv')
print('Physical Fourier rows:',len(FOURIER),'reference calibrations:',len(CALIBRATION))

Physical Fourier rows: 0 reference calibrations: 0


## Cell 67 — Reviewer-ready figures, model performance tables and claim-control register

Produce publication-ready **descriptive** visuals for measured original conductivity distributions, crossvalidated observed-vs-predicted scatter plots, residual plots, mean model errors, held-out groups, and diagnostics only where data exist. Export a **claim-control register** distinguishing reviewer requests solved computationally, blocked by missing measured evidence, and manuscript-only tasks (surface morphology, resin/hardener, curing, qualifications). Numerical reproducibility alone does not resolve those other reviewer criticisms.

In [34]:
fig,ax=plt.subplots(figsize=(7,4))
for reg,d in RAW.items():ax.hist(d[TARGET],bins=14,alpha=.5,label=f'{reg} (n={len(d)})')
ax.set_xlabel('Original recorded composite thermal conductivity (W/m·K)');ax.set_ylabel('Count');ax.legend();ax.set_title('Original literature dataset targets (heterogeneous materials)');save_figure('12_figures','original_target_distributions.png')
if not OOF_DF.empty:
    for reg,d in OOF_DF[(OOF_DF.augmentation=='none')].groupby('regime'):
        sel=d[d.model.eq(PREFERRED[reg])]
        if sel.empty:continue
        fig,ax=plt.subplots(figsize=(5.4,4.8));ax.scatter(sel.observed,sel.predicted,s=22,alpha=.8)
        lo=float(min(sel.observed.min(),sel.predicted.min()));hi=float(max(sel.observed.max(),sel.predicted.max()));ax.plot([lo,hi],[lo,hi],'k--',lw=1)
        ax.set_xlabel('Held-out observed k (W/m·K)');ax.set_ylabel('Held-out predicted k (W/m·K)');ax.set_title(reg+' — author-source held-out, provisional');save_figure('12_figures',f'{reg}_oof_observed_vs_predicted.png')
        fig,ax=plt.subplots(figsize=(5.4,4));ax.scatter(sel.predicted,sel.predicted-sel.observed,s=22,alpha=.8);ax.axhline(0,color='black',ls='--',lw=1);ax.set_xlabel('Held-out predicted k');ax.set_ylabel('Prediction error k (W/m·K)');ax.set_title(reg+' — original-row residuals');save_figure('12_figures',f'{reg}_oof_residuals.png')
CLAIMS=[
 ('E1','Five primary ML regressors plus optionally GPR/MLP with nested publication CV','PROVISIONAL' if not SCORES.empty else 'NOT_ESTIMABLE','Original rows; independently verified publications absent'),
 ('E2','Training-only Gaussian perturbation ablation','DONE_PROVISIONAL' if not AUG_RESULTS.empty else 'NOT_ESTIMABLE','Gaussian inputs are sensitivity analysis, not independent measurements'),
 ('E3','Material-system grouped CV','DONE' if MATERIAL_TEST else 'NOT_ESTIMABLE','Material-system identities missing'),
 ('E4','Evidence-verified epoxy-only vs broad','DONE' if EPOXY_CV else 'NOT_ESTIMABLE','Per-row evidence/loading type/experiment type missing'),
 ('E5','Effective-medium predictions','DONE' if len(PHYS_DF) else 'NOT_ESTIMABLE','Verified volume fraction and compatible morphology missing'),
 ('E6','Across-regime generalization','NOT_ESTIMABLE_CROSS_REGIME_TRANSFER','Unit/morphology/temperature and source metadata do not support cross-regime transfer'),
 ('E7','Split-conformal intervals','DONE_PROVISIONAL' if len(CONF_DF) else 'NOT_ESTIMABLE','Publication shift limits coverage guarantees'),
 ('E8','Independent external measurements and OOD','PROVISIONAL_CANDIDATES_OOD_ONLY' if EXT_SCORE else 'PARTIAL_OOD_ONLY','External measured source records missing'),
 ('E9','Measured versus HS bounds and constituent limits','DONE' if len(PHYS_DF) else 'FLAGS_ONLY','No verified unit/geometry per row'),
 ('E10','Fourier law, uncertainty and calibration','DONE_PARTIAL' if len(FOURIER) else 'NOT_ESTIMABLE','No actual heat-flow records, instrument calibration or reference standards'),
 ('R1_R2_R3','Resin/hardener curing chemistry, SEM, adhesion, space qualification','MANUSCRIPT_OR_LAB_ONLY','Cannot be generated by model training')]
save_csv([{'experiment':x,'requirement':y,'result_status':s,'limitation':lim} for x,y,s,lim in CLAIMS],'11_reports','scientific_claim_control_register.csv')
print('Saved claim-control matrix and final figures.')

Saved claim-control matrix and final figures.


### Supplement A — Add TabICLv2 without modifying the original models

This **additional** pretrained regression benchmark is evaluated on the original cryogenic and room-temperature held-out splits. The training-side only inner grouped CV chooses between ensemble sizes 2 and 4. It does **not** replace ExtraTrees, GradientBoosting, XGBoost, RandomForest or SVR and does not change any original result file. Internet is needed for the open TabICLv2 checkpoint download if uncached. Configure `TC_ENABLE_TABICL=1` (the default in full mode); Kaggle CPU is supported. If the pretrained checkpoint is unavailable, the notebook records an explicit failure and retains the five-model results.

In [35]:
# ADD-ON A — TabICLv2, kept separate from original five-model training.
# Original model definitions, hyperparameter search, ablations, registry and metrics remain intact.
# This cell runs AFTER the original training, BEFORE its unchanged final ZIP export.
import subprocess, importlib, gc, traceback
TABICL_ENABLED = os.getenv('TC_ENABLE_TABICL', '1' if RUN_MODE=='full' else '0') == '1'
TABICL_SEED = 20261108  # matches the earlier successful V4 reference experiment
TABICL_CANDIDATES = [2,4]  # V4 train-side ensemble search
TABICL_CHECKPOINT = 'tabicl-regressor-v2-20260212.ckpt'
TABICL_STATUS = []
TABICL_PRED = []
TABICL_FOLD = []
TABICL_INNER = []
TABICL_AVAILABLE = False
TABICL_PACKAGE_VERSION = None
TABICL_DEVICE = 'cpu'

def tabicl_status(regime, stage, state, reason):
    TABICL_STATUS.append({'regime':regime, 'stage':stage, 'status':state, 'details':str(reason)[:1500]})

def make_tabicl(n_ensemble, seed):
    return TabICLRegressor(n_estimators=int(n_ensemble), device=TABICL_DEVICE,
                           batch_size=2, random_state=int(seed),
                           checkpoint_version=TABICL_CHECKPOINT,
                           allow_auto_download=True)

def choose_tabicl_ensemble(X, y, groups, outer_train, fold, regime):
    # Crucial: inner score never uses any outer held-out rows.
    xt = X[outer_train]; yt = y[outer_train]; gt = np.asarray(groups)[outer_train]
    inner = list(GroupKFold(n_splits=2).split(xt, yt, gt))
    for aa,bb in inner:
        assert set(gt[aa]).isdisjoint(set(gt[bb]))
    tested=[]
    for n_ensemble in TABICL_CANDIDATES:
        actual=[]; prediction=[]
        for i,(aa,bb) in enumerate(inner):
            learner=make_tabicl(n_ensemble, TABICL_SEED+fold*37+i)
            learner.fit(xt[aa],yt[aa]); p=np.asarray(learner.predict(xt[bb]),float).ravel()
            assert len(p)==len(bb) and np.isfinite(p).all()
            actual.extend(yt[bb]);prediction.extend(p)
            del learner;gc.collect()
        score=float(mean_absolute_error(actual,prediction))
        tested.append((score,n_ensemble))
        TABICL_INNER.append({'regime':regime,'outer_fold':fold,'ensemble':n_ensemble,
                             'inner_mae':score,'inner_folds':2,'selection_scope':'OUTER_TRAIN_ONLY'})
    score,n_ensemble=min(tested)
    return int(n_ensemble),float(score)

if TABICL_ENABLED:
    try:
        try:
            import tabicl
        except ImportError:
            print('TabICL package missing; attempting to install tabicl>=2.2,<3.0 (requires Kaggle Internet).')
            cmd=[sys.executable,'-m','pip','install','-q','tabicl>=2.2,<3.0']
            result=subprocess.run(cmd, capture_output=True,text=True,timeout=360)
            if result.returncode:
                raise RuntimeError('tabicl installation failed: '+result.stderr[-1250:])
            importlib.invalidate_caches()
            import tabicl
        from tabicl import TabICLRegressor
        TABICL_PACKAGE_VERSION=getattr(tabicl,'__version__','unknown')
        try:
            import torch
            # Use a GPU only if one is already enabled; CPU is supported and reproduced the V4 result.
            TABICL_DEVICE='cuda' if torch.cuda.is_available() and os.getenv('TC_TABICL_GPU','0')=='1' else 'cpu'
        except ImportError:
            TABICL_DEVICE='cpu'
        TABICL_AVAILABLE=True
        tabicl_status('all','import','READY',f'tabicl={TABICL_PACKAGE_VERSION}, device={TABICL_DEVICE}')
    except Exception as exc:
        tabicl_status('all','import','FAILED',type(exc).__name__+': '+str(exc))
        print('TabICLv2 unavailable: original five-model training results are unaffected.')
else:
    tabicl_status('all','config','DISABLED','Set TC_ENABLE_TABICL=1 in Kaggle to enable the pretrained model')

if TABICL_AVAILABLE:
    for regime in ('cryogenic','room'):
        dataset=RAW[regime]
        X=dataset[FEATURES[regime]].to_numpy(dtype=float)
        y=dataset[TARGET].to_numpy(dtype=float)
        groups=dataset.publication_id.astype(str).to_numpy()
        for fold,(train,test) in enumerate(SPLITS[regime],1):
            try:
                assert set(groups[train]).isdisjoint(set(groups[test]))
                n_ens,inner_mae=choose_tabicl_ensemble(X,y,groups,train,fold,regime)
                learner=make_tabicl(n_ens,TABICL_SEED+fold*31)
                learner.fit(X[train],y[train])
                prediction=np.asarray(learner.predict(X[test]),float).ravel()
                assert len(prediction)==len(test) and np.isfinite(prediction).all()
                TABICL_FOLD.append({'regime':regime,'outer_fold':fold,
                    'model':'TabICLv2','n_estimators':n_ens,'inner_best_mae':inner_mae,
                    'validation_type':'PROVISIONAL_SOURCE_GROUPS',
                    **safe_metrics(y[test],prediction)})
                for j,pos in enumerate(test):
                    row=dataset.iloc[pos]
                    TABICL_PRED.append({'regime':regime,'model':'TabICLv2','orig_id':str(row.orig_id),
                        'publication_id':str(row.publication_id),'outer_fold':fold,
                        'observed':float(y[pos]),'predicted':float(prediction[j]),
                        'augmentation':'none','validation_type':'PROVISIONAL_SOURCE_GROUPS'})
                tabicl_status(regime,f'fold_{fold}','COMPLETE',f'ens={n_ens}, inner_mae={inner_mae:.5f}')
                print(f'TabICLv2 {regime} fold {fold}: MAE={mean_absolute_error(y[test],prediction):.5f}, ensemble={n_ens}')
                del learner;gc.collect()
            except Exception as exc:
                tabicl_status(regime,f'fold_{fold}','FAILED',type(exc).__name__+': '+str(exc))
                print(f'TabICLv2 FAILED: {regime} fold {fold}: {str(exc)[:240]}')
                # Never silently report partial folds as a valid out-of-fold benchmark.
                if any(key in str(exc).lower() for key in ('download','connection','permission','checkpoint','model path','network')):
                    break
save_csv(TABICL_STATUS,'11_reports','tabicl_execution_status.csv')
save_csv(TABICL_PRED,'05_nested_cv','tabicl_oof_predictions.csv')
save_csv(TABICL_FOLD,'05_nested_cv','tabicl_outer_fold_metrics.csv')
save_csv(TABICL_INNER,'05_nested_cv','tabicl_inner_ensemble_search.csv')


TabICL package missing; attempting to install tabicl>=2.2,<3.0 (requires Kaggle Internet).


Checkpoint 'tabicl-regressor-v2-20260212.ckpt' not cached.



tabicl-regressor-v2-20260212.ckpt: reconstructing file:   0%|          |  0.00B /  114MB            

tabicl-regressor-v2-20260212.ckpt: downloading bytes:           |  0.00B            

TabICLv2 cryogenic fold 1: MAE=0.09582, ensemble=4
TabICLv2 cryogenic fold 2: MAE=0.07844, ensemble=4
TabICLv2 cryogenic fold 3: MAE=0.07215, ensemble=2
TabICLv2 cryogenic fold 4: MAE=0.02718, ensemble=4
TabICLv2 cryogenic fold 5: MAE=0.04019, ensemble=2
TabICLv2 room fold 1: MAE=0.15725, ensemble=4
TabICLv2 room fold 2: MAE=0.16569, ensemble=2
TabICLv2 room fold 3: MAE=0.16193, ensemble=2
TabICLv2 room fold 4: MAE=0.17334, ensemble=4
TabICLv2 room fold 5: MAE=0.17012, ensemble=4


PosixPath('/kaggle/working/TC_BLOCK2_RESULTS/05_nested_cv/tabicl_inner_ensemble_search.csv')

### Supplement B — High-temperature exploratory R² and side-by-side score tables

High temperature has only **one declared publication**, so *independent-publication* R² cannot be obtained from these observations. For additional visibility, five-fold group CV uses numeric **composition signatures without temperature** to avoid sending identical compositions at different temperatures into both train and test. This is exploratory **same-source** interpolation and must not be reported as external validation. Outputs also include an exact-row-aligned TabICLv2 vs original Phase 2 comparison for cryogenic and room temperature.

In [36]:
# ADD-ON B — Exploratory high-temperature R², with NO false publication-independent claim.
# Keep all five original publication-held-out metrics unchanged (high: NOT ESTIMABLE).
# Group same numeric composition (excluding temperature) to prevent leakage across nearby temperatures.
# Also produce a side-by-side old-vs-TabICLv2 table for the two valid provisional evaluation regimes.
HIGH_EXP_PRED=[]; HIGH_EXP_FOLDS=[]; HIGH_EXP_SUMMARY=[]; ADDON_COMPARE=[]; TABICL_GROUP_COMPARE=[]
HIGH_D=RAW['high']; HIGH_X=HIGH_D[FEATURES['high']].to_numpy(float)
HIGH_Y=HIGH_D[TARGET].to_numpy(float)
composition_cols=['base_tc_w_mk','filler_tc_w_mk','filler_loading_as_recorded']
composition_signatures=HIGH_D[composition_cols].apply(lambda rec:'|'.join(format(float(x),'.12g') for x in rec),axis=1)
HIGH_GROUPS=pd.factorize(composition_signatures,sort=True)[0]
HIGH_NGROUPS=int(len(np.unique(HIGH_GROUPS)))
if HIGH_NGROUPS>=3:
    high_outer=list(GroupKFold(n_splits=min(5,HIGH_NGROUPS)).split(HIGH_X,HIGH_Y,HIGH_GROUPS))
    high_names=list(MODEL_NAMES)  # exactly the existing five conventional models, no original training changes
    if TABICL_AVAILABLE and RUN_MODE=='full':high_names.append('TabICLv2')
    for model_name in high_names:
        prior_count=len(HIGH_EXP_PRED)
        for fold,(tr,te) in enumerate(high_outer,1):
            assert not set(HIGH_GROUPS[tr]).intersection(set(HIGH_GROUPS[te]))
            try:
                if model_name=='TabICLv2':
                    n_ens,selection_score=choose_tabicl_ensemble(HIGH_X,HIGH_Y,HIGH_GROUPS,tr,fold,'high')
                    learner=make_tabicl(n_ens,TABICL_SEED+fold*31)
                else:
                    n_ens=None;selection_score=float('nan')
                    learner=model_factory(model_name,'none')  # existing model, unchanged hyperparameters
                learner.fit(HIGH_X[tr],HIGH_Y[tr])
                prediction=np.asarray(learner.predict(HIGH_X[te]),float).ravel()
                assert len(prediction)==len(te) and np.isfinite(prediction).all()
                HIGH_EXP_FOLDS.append({'regime':'high','model':model_name,'fold':fold,
                    'evaluation_type':'COMPOSITION_SIGNATURE_EXPLORATORY_ONLY_NOT_PUBLICATION_VALIDATION',
                    'n_train':len(tr),'n_test':len(te),'training_group_overlap':0,
                    'n_estimators_pretrained':n_ens,'inner_mae_pretrained':selection_score,
                    **safe_metrics(HIGH_Y[te],prediction)})
                for j,idx in enumerate(te):
                    HIGH_EXP_PRED.append({'regime':'high','model':model_name,'orig_id':str(HIGH_D.iloc[idx].orig_id),
                        'outer_fold':fold,'observed':float(HIGH_Y[idx]),'predicted':float(prediction[j]),
                        'composition_group':int(HIGH_GROUPS[idx]),
                        'evaluation_type':'COMPOSITION_SIGNATURE_EXPLORATORY_ONLY_NOT_PUBLICATION_VALIDATION'})
                del learner;gc.collect()
            except Exception as exc:
                tabicl_status('high',f'{model_name}_fold_{fold}','FAILED',type(exc).__name__+': '+str(exc))
                print(f'High exploratory FAILED {model_name} fold {fold}: {str(exc)[:220]}')
                break
        chunk=pd.DataFrame(HIGH_EXP_PRED[prior_count:])
        if len(chunk)==len(HIGH_D) and chunk.orig_id.nunique()==len(HIGH_D):
            m=safe_metrics(chunk.observed,chunk.predicted)
            HIGH_EXP_SUMMARY.append({'regime':'high','model':model_name,'augmentation':'none',
                **m,'n_composition_groups':HIGH_NGROUPS,
                'evaluation_type':'COMPOSITION_SIGNATURE_EXPLORATORY_ONLY_NOT_PUBLICATION_VALIDATION',
                'metrics_are_publication_independent':False})
        else:
            tabicl_status('high',f'{model_name}_completeness','INCOMPLETE',
                f'{len(chunk)}/{len(HIGH_D)} held-out predictions; no summary metric is reported')
else:
    tabicl_status('high','composition_groups','NOT_ESTIMABLE','Fewer than three distinct numeric compositions')

# These additions intentionally do not overwrite original SCIENTIFIC SCORES, OOF_DF, or REVIEWER_40_ISSUE_STATUS.
TBL_TABICL=pd.DataFrame(TABICL_PRED)
TBL_BASE=OOF_DF[OOF_DF.model.isin(MODEL_NAMES) & OOF_DF.augmentation.isin(AUGS)].copy()
for regime in ('cryogenic','room'):
    old=SCORES[(SCORES.regime==regime)&SCORES.model.isin(MODEL_NAMES)].sort_values('mae')
    if old.empty:continue
    winner=old.iloc[0]
    record={'regime':regime,'previous_best_model':winner.model,
            'previous_best_augmentation':winner.augmentation,
            'previous_best_mae':float(winner.mae),'previous_best_rmse':float(winner.rmse),
            'previous_best_r2':float(winner.r2),
            'tabicl_mae':float('nan'),'tabicl_rmse':float('nan'),'tabicl_r2':float('nan'),
            'delta_mae_tabicl_minus_previous':float('nan'),
            'percent_mae_reduction':float('nan'),'tabicl_complete':False,
            'validation_label':'PROVISIONAL_SOURCE_GROUPS_UNVERIFIED'}
    if not TBL_TABICL.empty:
        candidate=TBL_TABICL[TBL_TABICL.regime.eq(regime)]
        if len(candidate)==len(RAW[regime]) and candidate.orig_id.nunique()==len(RAW[regime]):
            prior=TBL_BASE[(TBL_BASE.regime==regime)&(TBL_BASE.model==winner.model)&
                           (TBL_BASE.augmentation==winner.augmentation)]
            matched=candidate[['orig_id','publication_id','observed','predicted']].merge(
                prior[['orig_id','publication_id','observed','predicted']],on=['orig_id','publication_id'],
                suffixes=('_tabicl','_previous'),validate='one_to_one')
            assert len(matched)==len(RAW[regime])
            assert np.allclose(matched.observed_tabicl,matched.observed_previous)
            m=safe_metrics(candidate.observed,candidate.predicted)
            record.update(tabicl_mae=m['mae'],tabicl_rmse=m['rmse'],tabicl_r2=m['r2'],
                delta_mae_tabicl_minus_previous=m['mae']-float(winner.mae),
                percent_mae_reduction=100*(float(winner.mae)-m['mae'])/float(winner.mae),
                tabicl_complete=True)
            group_maes=matched.assign(previous_err=(matched.observed_previous-matched.predicted_previous).abs(),
                                     tabicl_err=(matched.observed_tabicl-matched.predicted_tabicl).abs()).groupby('publication_id')[['previous_err','tabicl_err']].mean()
            differences=group_maes.tabicl_err-group_maes.previous_err
            TABICL_GROUP_COMPARE.append({'regime':regime,'n_provisional_groups':len(differences),
                'n_groups_improved':int((differences<0).sum()),
                'mean_group_delta_mae':float(differences.mean()),
                'two_sided_exact_signflip_p_UNADJUSTED':float(exact_signflip(differences.to_numpy())),
                'interpretation':'EXPLORATORY_UNVERIFIED_GROUPS_NO_INDEPENDENT_PUBLICATION_CLAIM'})
    ADDON_COMPARE.append(record)

ADDON_SUMMARY=pd.concat([SCORES[SCORES.model.isin(MODEL_NAMES)][['regime','model','augmentation','mae','rmse','r2']].assign(
                         evaluation_type='ORIGINAL_PROVISIONAL_PUBLICATION_GROUPS'),
                         pd.DataFrame([{'regime':q.regime,'model':'TabICLv2','augmentation':'none',
                                       'mae':q.tabicl_mae,'rmse':q.tabicl_rmse,'r2':q.tabicl_r2,
                                       'evaluation_type':'PROVISIONAL_SOURCE_GROUPS_UNVERIFIED'}
                                       for q in [type('R',(),record) for record in ADDON_COMPARE] if q.tabicl_complete]),
                         pd.DataFrame(HIGH_EXP_SUMMARY)[['regime','model','augmentation','mae','rmse','r2','evaluation_type']]
                            if HIGH_EXP_SUMMARY else pd.DataFrame(columns=['regime','model','augmentation','mae','rmse','r2','evaluation_type'])],
                        ignore_index=True)
save_csv(HIGH_EXP_PRED,'05_nested_cv','high_composition_exploratory_predictions.csv')
save_csv(HIGH_EXP_FOLDS,'05_nested_cv','high_composition_exploratory_fold_metrics.csv')
save_csv(HIGH_EXP_SUMMARY,'05_nested_cv','high_composition_exploratory_metrics.csv')
save_csv(TABICL_INNER,'05_nested_cv','tabicl_inner_ensemble_search.csv')  # include high-temperature inner settings too
save_csv(ADDON_COMPARE,'11_reports','phase2_original_vs_tabicl_comparison.csv')
save_csv(TABICL_GROUP_COMPARE,'06_stats','tabicl_vs_original_group_comparison.csv')
save_csv(ADDON_SUMMARY,'11_reports','phase2_all_models_with_exploratory_high.csv')
save_csv(TABICL_STATUS,'11_reports','tabicl_execution_status.csv')
run_info['tabicl_addon_enabled']=TABICL_ENABLED
run_info['tabicl_addon_package_version']=TABICL_PACKAGE_VERSION
run_info['tabicl_addon_checkpoint']=TABICL_CHECKPOINT
run_info['tabicl_addon_device']=TABICL_DEVICE
run_info['tabicl_addon_complete_regimes']=[x['regime'] for x in ADDON_COMPARE if x['tabicl_complete']]
run_info['high_exploratory_group_count']=HIGH_NGROUPS
run_info['high_exploratory_models_complete']=[x['model'] for x in HIGH_EXP_SUMMARY]
addon_text = ('# Supplement: TabICLv2 and high-temperature exploratory evaluation\n\n'
    '- The **original 70 notebook cells**, five-model pipelines, augmentation experiments, outputs, '\
    'and reviewer gates are not modified. These new supplementary files do not replace them.\n'
    '- Cryogenic and room TabICLv2 use the exact original outer publication splits and train-only two-fold ensemble selection. '\
    'Groups are **provisional and not independently paper-to-row verified**.\n'
    '- High temperature has **one declared publication**: publication-held-out metrics remain **not estimable**. '\
    'The new high R² is an exploratory 5-fold split by numeric base conductivity, filler conductivity and filler loading; '\
    f'{HIGH_NGROUPS} distinct composition signatures among {len(HIGH_D)} records. '\
    'Most compositions are unique, so this is a weak independence test within one publication.\n'
    '- The high conventional models use existing **default hyperparameters, no augmentation**, separately from the original final full-data models.\n'
    '- The pretrained checkpoint must be downloaded by Kaggle Internet or found in its cache. '\
    'If unavailable, failure is reported and no fake TabICLv2 score is produced.\n\n'
    '## Existing best vs TabICLv2 (MAE is W/(m·K))\n\n'
    +pd.DataFrame(ADDON_COMPARE).to_string(index=False)+'\n\n'
    '## High-temperature exploratory MAE, RMSE, R²\n\n'
    +pd.DataFrame(HIGH_EXP_SUMMARY).to_string(index=False)+'\n')
p=OUT/'11_reports'/'TABICL_AND_HIGH_EXPLORATORY_README.md';p.write_text(addon_text,encoding='utf-8');artifact(p,'11_reports')
print('\nMODEL EXTENSION COMPARISON\n',pd.DataFrame(ADDON_COMPARE).to_string(index=False))
print('\nHIGH TEMP — EXPLORATORY ONLY (NOT publication-level)\n',pd.DataFrame(HIGH_EXP_SUMMARY).to_string(index=False))
print('TabICLv2 completed regimes:',[x['regime'] for x in ADDON_COMPARE if x['tabicl_complete']])
print('Original model metric tables and publication held-out status remain unchanged.')



MODEL EXTENSION COMPARISON
    regime previous_best_model previous_best_augmentation  previous_best_mae  previous_best_rmse  previous_best_r2  tabicl_mae  tabicl_rmse  tabicl_r2  delta_mae_tabicl_minus_previous  percent_mae_reduction  tabicl_complete                     validation_label
cryogenic    GradientBoosting                       none           0.080351            0.117082           0.78995    0.063175     0.110137   0.814131                        -0.017176              21.376688             True PROVISIONAL_SOURCE_GROUPS_UNVERIFIED
     room          ExtraTrees              gaussian_1pct           0.196655            0.301245           0.64448    0.165667     0.278906   0.695254                        -0.030988              15.757728             True PROVISIONAL_SOURCE_GROUPS_UNVERIFIED

HIGH TEMP — EXPLORATORY ONLY (NOT publication-level)
 regime            model augmentation  n      mae     rmse       r2      bias  mape_pct  n_composition_groups                            

## Cell 69 — Completeness audit, 40-reviewer checklist, ZIP and run manifest

This final cell records reviewer **R1:17, R2:17, R3:6 = 40 comments**, maps every item to the relevant experiment/output or required manuscript/laboratory follow-up, and writes `REVIEWER_40_ISSUE_STATUS.csv`. It validates essential output files, numerical integrity and all actual model-file paths. It saves `RUN_MANIFEST.json`, `ARTIFACT_INDEX.csv`, `BLOCK2_FINAL_REPORT.md` and creates one **complete** `/kaggle/working/TC_BLOCK2_RESULTS.zip` without stale files. The notebook's terminal message lists any scientific blockers explicitly; a successful Python run **does not mean all scientific review points are resolved**.

In [37]:
REVIEWER_ISSUES = [
 ('R1',1,'Augmentation leakage','E2 / 05_nested_cv'),('R1',2,'Publication grouping','E3 / 03_splits'),('R1',3,'Nested tuning bias','E1 / 05_nested_cv'),('R1',4,'Paired statistics','E1 / 06_stats'),('R1',5,'Physical baselines/uncertainty','E5 E7 / 09_physics 10_uncertainty'),('R1',6,'Mixed material families','E4 / 02_preprocessing'),('R1',7,'Single-source high temperature','03_splits'),('R1',8,'259 versus 256 records','01_audit'),('R1',9,'Independent lab errors','E8 / 08_external'),('R1',10,'Theory versus experiment origin','02_preprocessing / 08_external'),('R1',11,'Discontinuous temperature coverage','01_audit'),('R1',12,'Uncertainty and calibration','E10 / 10_uncertainty'),('R1',13,'Plausibility/neat epoxy control','E5 E9 / 09_physics'),('R1',14,'Space qualification and adhesion','Manuscript/lab evidence'),('R1',15,'Feature interpretation non-causal','07_interpretation'),('R1',16,'Reproducibility/data sharing','00_config / 13_packages'),('R1',17,'Title numbering citation editing','Manuscript revision'),
 ('R2',1,'Augmentation partition ordering','E2 / 05_nested_cv'),('R2',2,'Material/publication CV','E3 / 03_splits'),('R2',3,'Detailed provenance/epoxy content','01_audit / 02_preprocessing'),('R2',4,'Wide-range temperature claims','01_audit'),('R2',5,'CNT context and network morphology','Manuscript/lab evidence'),('R2',6,'Resin hardener curing specifics','Lab/manuscript'),('R2',7,'Independent specimen counts/error bars','Lab record request'),('R2',8,'SEM porosity dispersion','Lab/manuscript'),('R2',9,'Measured conductivity uncertainty','E10 / 10_uncertainty'),('R2',10,'Fourier steady-state heat flow','E10 / 10_uncertainty'),('R2',11,'Cryogenic constituent plausibility','E5 E9 / 09_physics'),('R2',12,'Graphene morphology interactions','Manuscript/lab'),('R2',13,'Physically consistent augmentation','E2 / 04_models'),('R2',14,'Feature importance causality','07_interpretation'),('R2',15,'Model performance statistical tests','06_stats'),('R2',16,'Theory versus experiment separation','02_preprocessing / 09_physics'),('R2',17,'Unseen sample error table','E8 / 08_external'),
 ('R3',1,'English revision','Manuscript editing'),('R3',2,'Augmentation leakage','E2 / 05_nested_cv'),('R3',3,'Grouped source/material','E3 / 03_splits'),('R3',4,'Measurement uncertainty','E10 / 10_uncertainty'),('R3',5,'Paired comparisons and lab errors','06_stats / 08_external'),('R3',6,'Suggested new literature','Manuscript bibliography')]
assert len(REVIEWER_ISSUES)==40
CHECK=[]
for reviewer,n,issue,where in REVIEWER_ISSUES:
    implemented_topics={'Augmentation leakage','Publication grouping','Nested tuning bias','Paired statistics',
        'Augmentation partition ordering','Material/publication CV','Grouped source/material'}
    state=('COMPUTED_PROVISIONAL_UNVERIFIED_GROUPS' if issue in implemented_topics and RUN_MODE=='full'
          else 'SMOKE_NOT_FINAL' if issue in implemented_topics else 'CONDITIONAL_OR_MANUSCRIPT_LAB_FOLLOWUP')
    CHECK.append({'reviewer':reviewer,'comment_number':n,'topic':issue,'planned_evidence_or_step':where,'status':state,'does_not_imply_reviewer_resolved':True})
save_csv(CHECK,'11_reports','REVIEWER_40_ISSUE_STATUS.csv')
assert sum(len(z) for z in RAW.values())==256
assert AUGMENTATION_COPIES==2, 'Training-only input sensitivity must use two synthetic copies for 3x ablation'
assert not SPLITS['high'] and len(FINAL_REGISTRY_DF)==len(MODEL_NAMES)*len(AUGS)*3
if not OOF_DF.empty:
    assert OOF_DF.groupby(['regime','grouping','orig_id','model','augmentation']).size().max()==1
    assert np.isfinite(OOF_DF[['observed','predicted']].to_numpy(dtype=float)).all()
if RUN_MODE=='full':assert set(MODEL_NAMES)=={'ExtraTrees','GradientBoosting','XGBoost','RandomForest','SVR'}
mandatory=['01_audit/input_file_hashes.csv','01_audit/schema_validation.csv','01_audit/row_quality_flags.csv','02_preprocessing/eligibility_ledger.csv','03_splits/publication_outer_split_manifest.csv','04_models/final_model_registry.csv','05_nested_cv/outer_fold_metrics.csv','05_nested_cv/outer_predictions.csv','06_stats/augmentation_paired_ablation.csv','07_interpretation/heldout_group_permutation_importance.csv','08_external/external_predictions.csv','09_physics/effective_medium_unit_tests.csv','10_uncertainty/heldout_group_interval_coverage.csv','10_uncertainty/fourier_law_measurements.csv','11_reports/REVIEWER_40_ISSUE_STATUS.csv']
missing=[x for x in mandatory if not (OUT/x).is_file()]
if missing:raise AssertionError('Mandatory output files missing: '+str(missing))
for r in FINAL_REGISTRY_DF.itertuples():
    fp=OUT/r.model_relative_path
    assert fp.is_file() and hashlib.sha256(fp.read_bytes()).hexdigest()==r.sha256
# Capture the final status as a stable table before constructing the registry.
save_csv(STATES,'11_reports','FINAL_EXPERIMENT_STATUS.csv')
run_info.update({'original_record_count':256,'regime_counts':{k:len(v) for k,v in RAW.items()},'publication_counts':{k:int(v.publication_id.nunique()) for k,v in RAW.items()},'source_labels':'AUTHOR_CONFIRMED_SAME_AS_PREVIOUSLY_RANDOM_ASSIGNED_NOT_INDEPENDENTLY_MATCHED','scientific_signoff':'BLOCKED: SOURCE_LABELS_RANDOM_ORIGIN_UNVERIFIED; UNITS_AND_EXPERIMENTS_MISSING','n_model_files':len(FINAL_REGISTRY_DF),'optional_gpr_mlp_baselines_enabled':EXTRA_BASELINES,'n_augmentation_variants':len(AUGS),'historical_3pct_tested':('gaussian_3pct_historical' in AUGS),'cross_publication_exact_feature_overlap_rows':len(CROSS_GROUP_OVERLAPS),'n_oof_predictions':len(OOF_DF),'n_independent_external':0,'n_physics_eligible':len(PHYS_DF),'n_calibration_rows':len(CALIBRATION),'reviewer_comments_tracked':len(CHECK)})
save_json(run_info,'00_config','RUN_MANIFEST.json')
rows=list({x['relative_path']:x for x in ARTIFACTS}.values())
save_csv(rows,'11_reports','ARTIFACT_INDEX.csv')
report=f"""# BLOCK 2 — Execution report\n\n- Execution mode: **{RUN_MODE}**. Smoke tests do not qualify as journal results.\n- Original observation counts: cryogenic=54, room=160, high=42; total 256. Manuscript claims 259; 3 are unresolved.\n- Source IDs: **author-confirmed but identical to earlier random placeholder assignments and NOT paper-to-row verified**; grouping performance is software-only exploratory.\n- Grouped publication outer results: {len(FOLD_DF)} fold metric rows and {len(OOF_DF)} row-level prediction records.\n- Final saved models: {len(FINAL_REGISTRY_DF)}. High models may be fit but their grouped performance is **NOT ESTIMABLE** (one publication).\n- Strict epoxy cohort: {int(DATA.strict_epoxy_eligible.sum())} eligible rows. Material-system group CV: {len(MATERIAL_TEST)} outer folds.\n- Physical effective-medium eligible comparisons: {len(PHYS_DF)}; missing volume units and morphology gate other observations.\n- External independently verified rows: **0**; author-supplied external metadata alone never proves independence.\n- Fourier-law measurement rows: {len(FOURIER)}, calibration records: {len(CALIBRATION)}.\n- Forty reviewer topics tracked from the available issue register (the original 8-page reviewer PDF and 29-page manuscript were directly inspected in the final audit; the reported ML results remain provisional); computed outputs do not substitute fabrication, calibration, SEM or manuscript corrections.\n- Models, stage-wise CSVs, runtime hashes, per-fold tuning tables and figures are archived.\n\n## Critical interpretation\n\nDo not report the generated models as experimentally verified epoxy-specific space-grade predictors unless all independent source/material/measurement issues are resolved. Training scores and scientific signoff are different gates.\n"""
p=OUT/'11_reports'/'BLOCK2_FINAL_REPORT.md';p.write_text(report,encoding='utf-8');artifact(p,'11_reports')
# Avoid circular ARTIFACT_INDEX hash of itself. Final index includes all OTHER files only.
rows=list({x['relative_path']:x for x in ARTIFACTS if x['relative_path']!='11_reports/ARTIFACT_INDEX.csv'}.values())
save_csv(rows,'11_reports','ARTIFACT_INDEX.csv')
from zipfile import ZipFile, ZIP_DEFLATED
zip_path=WORKING/'TC_BLOCK2_RESULTS.zip'
with ZipFile(zip_path,'w',ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():z.write(f,f.relative_to(WORKING))
with ZipFile(zip_path) as z:
    assert z.testzip() is None
    names=z.namelist()
    assert 'TC_BLOCK2_RESULTS/04_models/final_model_registry.csv' in names
    assert 'TC_BLOCK2_RESULTS/11_reports/BLOCK2_FINAL_REPORT.md' in names
print('BLOCK 2 FINISHED. File:',zip_path,'size MB:',round(zip_path.stat().st_size/1e6,2))
print('Run mode:',RUN_MODE,' | Original rows: 256 | Final model files:',len(FINAL_REGISTRY_DF),'| Reviewer topics:',len(CHECK))
print('Scientific signoff remains conditional; see 11_reports/BLOCK2_FINAL_REPORT.md')
# Emit immutable, machine-readable model/environment/system evidence gates to complement the report.
print('PHASE 2 SCIENTIFIC GATE: BLOCKED until row-to-paper provenance, loading units, material identity and actual lab records are verified.')


BLOCK 2 FINISHED. File: /kaggle/working/TC_BLOCK2_RESULTS.zip size MB: 23.11
Run mode: full  | Original rows: 256 | Final model files: 45 | Reviewer topics: 40
Scientific signoff remains conditional; see 11_reports/BLOCK2_FINAL_REPORT.md
PHASE 2 SCIENTIFIC GATE: BLOCKED until row-to-paper provenance, loading units, material identity and actual lab records are verified.
